# 0. Setup & Data Loading

## 0.1 Load Data

In [ ]:
# 1. Imports and reusable project utilities
from datetime import date
from pathlib import Path
import hashlib
import json
import os
import platform

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import sklearn
import xgboost
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from utils.model_split import data_split
from utils.model_preprocessing import build_preprocessor
from utils.model_experiment import run_experiment
from utils.model_assessment import display_initial_assessment

# 2. Project paths and independent development configuration
project_root = Path.cwd().resolve().parents[1]
if not (project_root / "config/development_config.json").is_file():
    raise FileNotFoundError("Run the notebook from pipelines/model: config/development_config.json is missing")
data_root = Path(
    os.environ.get(
        "DATA_ROOT",
        project_root / "data",
    )
).resolve()
gold_root = data_root / "datamart" / "gold"
experiment_root = project_root / "artifacts/development/experiments"
development_config = json.loads(
    (project_root / "config/development_config.json").read_text(encoding="utf-8")
)
split_id = development_config["split_id"]
split_dir = project_root / "artifacts/development/splits" / split_id
split_config = development_config["split_calendar"]
feature_contract_path = gold_root / "feature_contract.json"
assert feature_contract_path.exists(), feature_contract_path

# 3. Scan Gold model data; only Development has predictors and target
model_parts = []
label_parts = []
source_files = [feature_contract_path]
test_start = date.fromisoformat(split_config["test_start"])
test_end = date.fromisoformat(split_config["test_end"])

for hotel in ("H1", "H2"):
    model_path = gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet"
    label_path = gold_root / "labels" / f"gold_label_store_{hotel}.parquet"
    source_files.extend(sorted(model_path.glob("*.parquet")))
    source_files.extend(sorted(label_path.glob("*.parquet")))

    model_dataset = ds.dataset(model_path, format="parquet")
    # Check out-of-calendar metadata without reading Test outcomes.
    out_of_range = model_dataset.to_table(
        columns=["Booking_ID"], filter=ds.field("CutoffDate") >= test_end
    )
    missing_cutoff = model_dataset.to_table(
        columns=["Booking_ID"], filter=ds.field("CutoffDate").is_null()
    )
    assert out_of_range.num_rows == 0 and missing_cutoff.num_rows == 0
    development_data = model_dataset.to_table(
        filter=ds.field("CutoffDate") < test_start
    ).to_pandas()
    test_metadata = model_dataset.to_table(
        columns=["Hotel", "Booking_ID", "CutoffDate", "ArrivalDate", "TotalNights"],
        filter=(ds.field("CutoffDate") >= test_start)
        & (ds.field("CutoffDate") < test_end),
    ).to_pandas()
    test_metadata["IsCanceled"] = pd.NA
    model_parts.extend([development_data, test_metadata])

    # Filter by known Development IDs before materializing outcome dates.
    label_dataset = ds.dataset(label_path, format="parquet")
    label_dates = label_dataset.to_table(
        columns=["Hotel", "Booking_ID", "IsCanceled", "ReservationStatusDate"],
        filter=ds.field("Booking_ID").isin(development_data.Booking_ID.tolist()),
    ).to_pandas()
    label_parts.append(label_dates)

# 4. Contract and input checks
feature_contract = json.loads(feature_contract_path.read_text(encoding="utf-8"))
model_data = pd.concat(model_parts, ignore_index=True)
label_dates = pd.concat(label_parts, ignore_index=True)
keys = ["Hotel", "Booking_ID"]
assert not model_data.duplicated(keys).any()
assert not label_dates.duplicated(keys).any()
assert len(label_dates) == model_data.IsCanceled.notna().sum()
assert set(model_data.loc[model_data.IsCanceled.notna(), "IsCanceled"]) == {0, 1}
assert model_data.loc[model_data.IsCanceled.isna(), "CutoffDate"].min() >= test_start
assert set(feature_contract["numeric_predictors"] + feature_contract["categorical_predictors"]).issubset(
    model_data.columns
)

source_identity = [
    {"path": str(path.relative_to(gold_root)), "size": path.stat().st_size,
     "mtime_ns": path.stat().st_mtime_ns}
    for path in source_files
]
source_fingerprint = hashlib.sha256(
    json.dumps(source_identity, sort_keys=True).encode("utf-8")
).hexdigest()
print("Gold:", gold_root)
print("Development labels:", len(label_dates))
print("Test metadata only:", int(model_data.IsCanceled.isna().sum()))
print("Source fingerprint:", source_fingerprint)
print("Runtime:", platform.python_version(), pd.__version__, sklearn.__version__, xgboost.__version__)

## 0.2 Data Split

In [ ]:
# 1. Split configuration and source identity
saved_split_config = {
    "schema_version": 2,
    "split_id": split_id,
    "calendar": split_config,
    "training_eligibility": development_config["training_eligibility"],
    "source_fingerprint": source_fingerprint,
    "source_identity": source_identity,
    "label_rule": "checkout=max(status date, arrival+total nights); canceled=status date",
}
expected_split_files = {"config.json", "membership.parquet", "audit.json"}

# 2. Load or generate the one versioned master split without overwriting it
if split_dir.exists():
    assert {path.name for path in split_dir.iterdir()} == expected_split_files
    saved_config = json.loads((split_dir / "config.json").read_text(encoding="utf-8"))
    for field in (
        "schema_version", "split_id", "calendar",
        "training_eligibility", "label_rule",
    ):
        assert saved_config[field] == saved_split_config[field], (
            f"Saved split {field} differs; use a new split ID"
        )

    # File timestamps and path separators can change across operating systems.
    # Compare stable paths/sizes and exact booking membership instead.
    def portable_manifest(entries):
        return sorted(
            (entry["path"].replace(chr(92), "/"), entry["size"])
            for entry in entries
        )

    assert portable_manifest(saved_config["source_identity"]) == portable_manifest(
        source_identity
    ), "Gold source files differ; inspect inputs and use a new split ID"
    membership = pd.read_parquet(split_dir / "membership.parquet")
    split_audit = json.loads((split_dir / "audit.json").read_text(encoding="utf-8"))
    current_membership, current_audit = data_split(
        model_data=model_data,
        label_dates=label_dates,
        split_config=split_config,
    )
    pd.testing.assert_frame_equal(
        membership.sort_values(keys).reset_index(drop=True),
        current_membership.sort_values(keys).reset_index(drop=True),
        check_dtype=False,
    )
    assert split_audit == current_audit
    split_status = "loaded and revalidated"
else:
    membership, split_audit = data_split(
        model_data=model_data,
        label_dates=label_dates,
        split_config=split_config,
    )
    split_dir.parent.mkdir(parents=True, exist_ok=True)
    temporary_split = split_dir.with_name(f".{split_id}.tmp")
    assert not temporary_split.exists(), f"Inspect incomplete split: {temporary_split}"
    temporary_split.mkdir()
    (temporary_split / "config.json").write_text(
        json.dumps(saved_split_config, indent=2) + "\n",
        encoding="utf-8",
    )
    membership.to_parquet(temporary_split / "membership.parquet", index=False)
    (temporary_split / "audit.json").write_text(
        json.dumps(split_audit, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_split.rename(split_dir)
    split_status = "created"

# 3. Validate partition, four validation windows and Rule B at every fit
assert len(membership) == len(model_data)
assert not membership.duplicated(keys).any()
assert membership.loc[membership.Partition.eq("Test"), "ValFold"].isna().all()
assert membership.loc[membership.Partition.eq("Test"), "LabelAvailableDate"].isna().all()
assert (membership.ArrivalDate - membership.CutoffDate).eq(
    pd.Timedelta(days=14)
).all()
assert membership.ExpectedCheckoutDate.ge(membership.ArrivalDate).all()
for fold_number, fold in enumerate(split_config["folds"], start=1):
    fit_date = pd.Timestamp(fold["fit_date"])
    train = membership.loc[
        membership.Partition.eq("Dev")
        & membership.CutoffDate.lt(fit_date)
        & membership.LabelAvailableDate.lt(fit_date)
        & membership.ExpectedCheckoutDate.lt(fit_date)
    ]
    validation = membership.loc[
        membership.ValFold.eq(fold_number).fillna(False)
    ]
    assert len(train) == split_audit["folds"][fold_number - 1]["train_eligible"]
    assert validation.CutoffDate.ge(fit_date).all()
    assert validation.CutoffDate.lt(pd.Timestamp(fold["val_end"])).all()
    assert validation.LabelAvailableDate.lt(
        pd.Timestamp(split_config["test_start"])
    ).all()

refit_date = pd.Timestamp(split_config["test_start"])
expected_refit = (
    membership.Partition.eq("Dev")
    & membership.CutoffDate.lt(refit_date)
    & membership.LabelAvailableDate.lt(refit_date)
    & membership.ExpectedCheckoutDate.lt(refit_date)
)
assert membership.FinalRefit.astype(bool).equals(expected_refit)

# 4. Display split summary
print("Split:", split_id, split_status, "-", len(membership), "bookings")
display(pd.DataFrame(split_audit["by_hotel"]).T)
display(pd.DataFrame(split_audit["folds"]))

# 1. Experiment Configuration

In [ ]:
# 1. Derive all 30 predictors directly from the Gold feature contract
binary_features = development_config["binary_predictors"]
numeric_features = [
    name for name in feature_contract["numeric_predictors"]
    if name not in binary_features
]
selected_features = {
    "numeric": numeric_features,
    "binary": binary_features,
    "categorical": feature_contract["categorical_predictors"],
}
feature_columns = [
    name
    for group in ("numeric", "binary", "categorical")
    for name in selected_features[group]
]

# 2. Validate exact coverage, role assignment and forbidden inputs
assert len(feature_columns) == len(set(feature_columns)) == 30
assert set(feature_columns) == set(
    feature_contract["numeric_predictors"]
    + feature_contract["categorical_predictors"]
)
assert set(binary_features).issubset(feature_contract["numeric_predictors"])
assert set(feature_columns).issubset(model_data.columns)
for model_name in ("Dummy", "LR", "RF", "XGB"):
    build_preprocessor(selected_features, feature_contract, model_name)

# 3. Define the four fixed initial model configurations
model_parameters = development_config["models"]
model_definitions = {
    "Dummy": DummyClassifier(**model_parameters["Dummy"]),
    "LR": LogisticRegression(**model_parameters["LR"]),
    "RF": RandomForestClassifier(**model_parameters["RF"]),
    "XGB": XGBClassifier(**model_parameters["XGB"]),
}

# 4. Build exactly one run per hotel and model
experiment_version = development_config["experiment_version"]
experiment_registry = [
    {
        "hotel": hotel,
        "model_name": model_name,
        "experiment_id": f"{hotel}_{model_name}_v{experiment_version}",
        "features": selected_features,
    }
    for hotel in ("H1", "H2")
    for model_name in ("Dummy", "LR", "RF", "XGB")
]
assert len(experiment_registry) == 8
assert len({item["experiment_id"] for item in experiment_registry}) == 8

# 5. Display the experiment plan and the inherited availability assumption
display(pd.DataFrame([
    {
        "Hotel": item["hotel"],
        "Model": item["model_name"],
        "Predictors": len(feature_columns),
        "Experiment ID": item["experiment_id"],
        "Split": split_id,
    }
    for item in experiment_registry
]))
print(development_config["availability_assumption"])

# 2. Initial Experiments

## 2.1 Execute Experiments

In [ ]:
# 1. Prepare execution and the exact saved artifact contract
expected_files = {"config.json", "metrics.parquet", "predictions.parquet"}
run_status = []

# 2. Load complete matching runs or fit and save new runs
for entry in experiment_registry:
    experiment_id = entry["experiment_id"]
    model_name = entry["model_name"]
    run_dir = experiment_root / experiment_id
    expected_params = {
        name: "NaN" if isinstance(value, float) and np.isnan(value) else value
        for name, value in model_definitions[model_name].get_params().items()
    }
    if run_dir.exists():
        assert {path.name for path in run_dir.iterdir()} == expected_files
        saved = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
        assert saved["split_id"] == split_id
        assert saved["experiment_version"] == experiment_version
        assert saved["hotel"] == entry["hotel"]
        assert saved["model_name"] == model_name
        assert saved["selected_features"] == selected_features
        assert saved["estimator_params"] == expected_params
        assert "ExpectedCheckoutDate < FitDate" in saved["training_rule"]
        status = "loaded"
    else:
        preprocessor = build_preprocessor(
            selected_features=selected_features,
            feature_contract=feature_contract,
            model_name=model_name,
        )
        run_experiment(
            model_data=model_data,
            membership=membership,
            hotel=entry["hotel"],
            selected_features=selected_features,
            estimator=model_definitions[model_name],
            preprocessor=preprocessor,
            experiment_id=experiment_id,
            split_id=split_id,
            split_config=split_config,
            output_root=experiment_root,
            experiment_version=experiment_version,
        )
        status = "created"
    run_status.append({
        "Hotel": entry["hotel"],
        "Model": model_name,
        "Experiment ID": experiment_id,
        "Status": status,
    })

# 3. Display load/create status for all eight runs
assert len(run_status) == 8
display(pd.DataFrame(run_status))

## 2.2 Verify Experiment Outputs

In [ ]:
# 1. Verify complete artifacts and the 30-predictor configurations
from sklearn.metrics import log_loss, roc_auc_score

verification_rows = []
reference_cohorts = {}
test_keys = set(map(
    tuple,
    membership.loc[membership.Partition.eq("Test"), keys].to_numpy(),
))
for entry in experiment_registry:
    run_dir = experiment_root / entry["experiment_id"]
    assert {path.name for path in run_dir.iterdir()} == expected_files
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    metrics = pd.read_parquet(run_dir / "metrics.parquet")
    predictions = pd.read_parquet(run_dir / "predictions.parquet")
    assert config["split_id"] == split_id
    assert config["feature_count"] == 30
    assert config["selected_features"] == selected_features
    assert config["folds"] == split_config["folds"]
    assert len(metrics) == 8
    assert metrics.groupby("split").size().to_dict() == {"train": 4, "val": 4}

    # 2. Verify Rule B Train counts/labels and unchanged Validation membership
    for fold_number, fold in enumerate(split_config["folds"], start=1):
        fit_date = pd.Timestamp(fold["fit_date"])
        expected_train = membership.loc[
            membership.Hotel.eq(entry["hotel"])
            & membership.Partition.eq("Dev")
            & membership.CutoffDate.lt(fit_date)
            & membership.LabelAvailableDate.lt(fit_date)
            & membership.ExpectedCheckoutDate.lt(fit_date),
            keys,
        ]
        train_score = metrics.loc[
            metrics.fold.eq(fold_number) & metrics.split.eq("train")
        ].iloc[0]
        expected_positives = expected_train.merge(
            model_data[keys + ["IsCanceled"]],
            on=keys,
            validate="one_to_one",
        ).IsCanceled.sum()
        assert train_score.n_bookings == len(expected_train)
        assert train_score.n_positive == expected_positives

    expected_val = membership.loc[
        membership.Hotel.eq(entry["hotel"]) & membership.ValFold.notna(),
        ["Booking_ID", "ValFold", "CutoffDate", "ArrivalDate"],
    ].rename(columns={"ValFold": "fold", "ArrivalDate": "StayDate"})
    assert len(predictions) == len(expected_val)
    joined = predictions.merge(
        expected_val,
        on=["Booking_ID", "fold"],
        validate="one_to_one",
        suffixes=("", "_expected"),
    )
    assert len(joined) == len(expected_val)
    assert joined.CutoffDate.eq(joined.CutoffDate_expected).all()
    assert joined.StayDate.eq(joined.StayDate_expected).all()
    expected_labels = predictions[["Hotel", "Booking_ID", "y_true"]].merge(
        model_data[keys + ["IsCanceled"]],
        on=keys,
        validate="one_to_one",
    )
    assert expected_labels.y_true.eq(expected_labels.IsCanceled).all()

    # 3. Verify OOF probabilities, cohort alignment, no Test keys and metrics
    assert predictions.Booking_ID.notna().all()
    assert not predictions.Booking_ID.duplicated().any()
    assert predictions.p_cancel.between(0, 1).all()
    assert np.isfinite(predictions.p_cancel).all()
    assert not set(map(tuple, predictions[keys].to_numpy())).intersection(test_keys)
    cohort = predictions[
        ["Booking_ID", "fold", "y_true", "CutoffDate", "StayDate"]
    ].sort_values(["Booking_ID", "fold"]).reset_index(drop=True)
    if entry["hotel"] in reference_cohorts:
        pd.testing.assert_frame_equal(cohort, reference_cohorts[entry["hotel"]])
    else:
        reference_cohorts[entry["hotel"]] = cohort

    for fold_number in range(1, 5):
        scored = predictions.loc[predictions.fold.eq(fold_number)]
        score = metrics.loc[
            metrics.fold.eq(fold_number) & metrics.split.eq("val")
        ].iloc[0]
        daily = scored.groupby("StayDate").agg(
            predicted=("p_cancel", "sum"),
            actual=("y_true", "sum"),
        )
        error = daily.predicted - daily.actual
        assert np.isclose(
            score.log_loss,
            log_loss(scored.y_true, scored.p_cancel, labels=[0, 1]),
        )
        assert np.isclose(
            score.roc_auc,
            roc_auc_score(scored.y_true, scored.p_cancel),
        )
        assert np.isclose(
            score.daily_bias_pct,
            100 * error.sum() / daily.actual.sum(),
        )
        assert np.isclose(
            score.daily_nrmse,
            np.sqrt(np.mean(error ** 2)) / daily.actual.mean(),
        )
        assert score.n_days == len(daily)

    verification_rows.append({
        "Hotel": entry["hotel"],
        "Model": entry["model_name"],
        "Experiment ID": entry["experiment_id"],
        "Folds": 4,
        "Validation Bookings": len(predictions),
        "Status": "verified",
    })

# 4. Display verification summary only
assert len(verification_rows) == 8
display(pd.DataFrame(verification_rows))

## 2.3 Initial Model Assessment

In [ ]:
# Generate all four assessment outputs from saved artifacts.
# No model training is performed in this stage.
display_initial_assessment(
    experiment_root=experiment_root,
    experiment_registry=experiment_registry,
    daily_fit_model="XGB",  # Daily Fit uses all four OOF folds by default.
)

### Initial Model Assessment — Key Insights

- **Model Performance:** RF outperforms Dummy on Log Loss for both hotels, but H2 shows no improvement in daily forecasting. → **Hypothesis:** Better booking-level probabilities do not necessarily translate into more accurate daily cancellation estimates.

- **Temporal Stability:** H2 RF underperforms in Fold 1 but improves in later folds. → **Hypothesis:** Changes in cancellation behavior, booking composition, or limited early training history may affect model performance.

- **Probability Reliability:** RF tends to overpredict cancellation risk, particularly for H2 bookings in the 20–40% probability range. → **Hypothesis:** Probability errors may be concentrated in specific risk groups or booking segments.

- **Generalization:** RF and XGB show noticeable Train–Validation Log Loss gaps, especially for H2. → **Hypothesis:** Model complexity or temporal distribution shifts may limit generalization.

- **Daily Forecasting:** Daily predictions show both systematic deviations and occasional large errors. → **Hypothesis:** Probability bias, seasonality, or booking composition may explain the remaining daily forecasting errors.

- **Model Comparison:** XGB achieves slightly higher ROC-AUC than RF but worse Log Loss, while LR shows unstable performance across folds. → **Hypothesis:** Models differ in probability estimation quality and sensitivity to training conditions.

# 3. Diagnostic Analysis

## 3.1 Temporal Drift Analysis

**Question:** Do the fold-level rates, completed historical cohorts, and booking composition help explain H2 RF's Fold 1 overprediction?

In [ ]:
# Load only Development Gold rows and saved split_v2 OOF artifacts. Validate
# identity and one-to-one booking alignment before any diagnostic display.
import matplotlib.pyplot as plt
from IPython.display import HTML, display

diagnostic_membership = ds.dataset(
    split_dir / "membership.parquet", format="parquet"
).to_table(filter=ds.field("Partition") == "Dev").to_pandas()
diagnostic_columns = keys + [
    "IsCanceled", "MarketSegment", "CustomerType",
    "LeadTime", "ArrivalDateMonth",
]
diagnostic_gold = pd.concat([
    ds.dataset(
        gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet",
        format="parquet",
    ).to_table(
        columns=diagnostic_columns,
        filter=ds.field("CutoffDate") < pd.Timestamp(split_config["test_start"]).date(),
    ).to_pandas()
    for hotel in ("H1", "H2")
], ignore_index=True)
assert not diagnostic_gold.duplicated(keys).any()
diagnostic_dev = diagnostic_membership.merge(
    diagnostic_gold, on=keys, validate="one_to_one"
)
assert len(diagnostic_dev) == len(diagnostic_membership)
assert diagnostic_dev.IsCanceled.notna().all()

diagnostic_predictions = {}
diagnostic_metrics = {}
diagnostic_configs = {}
expected_validation = diagnostic_dev.loc[
    diagnostic_dev.ValFold.notna(),
    keys + ["ValFold", "IsCanceled", "CutoffDate", "ArrivalDate"],
].rename(columns={"ValFold": "fold"})
for entry in experiment_registry:
    hotel, model = entry["hotel"], entry["model_name"]
    run_dir = experiment_root / entry["experiment_id"]
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    predictions = pd.read_parquet(run_dir / "predictions.parquet")
    metrics = pd.read_parquet(run_dir / "metrics.parquet")
    assert config["split_id"] == "split_v2"
    assert config["feature_count"] == 30
    assert config["selected_features"] == selected_features
    assert config["experiment_id"] == entry["experiment_id"]
    assert config["hotel"] == hotel and config["model_name"] == model
    assert predictions.split_id.eq("split_v2").all()
    assert predictions.Hotel.eq(hotel).all()
    assert predictions.model_name.eq(model).all()
    assert predictions.experiment_id.eq(entry["experiment_id"]).all()
    assert not predictions.duplicated(keys).any()
    assert predictions.p_cancel.between(0, 1).all()
    expected = expected_validation.loc[expected_validation.Hotel.eq(hotel)]
    aligned = predictions.merge(
        expected, on=keys, how="inner", validate="one_to_one",
        suffixes=("", "_expected"),
    )
    assert len(aligned) == len(predictions) == len(expected)
    assert aligned.fold.eq(aligned.fold_expected).all()
    assert aligned.y_true.eq(aligned.IsCanceled).all()
    assert aligned.CutoffDate.eq(aligned.CutoffDate_expected).all()
    assert aligned.StayDate.eq(aligned.ArrivalDate).all()
    assert set(predictions.fold) == {1, 2, 3, 4}
    diagnostic_predictions[(hotel, model)] = predictions
    diagnostic_metrics[(hotel, model)] = metrics
    diagnostic_configs[(hotel, model)] = config

# Complete Follow-up Rule is applied at each FitDate. Validation retains
# its full assigned cohort; rate differences alone do not establish drift.
fold_rate_rows = []
history_rows = []
for hotel in ("H1", "H2"):
    hotel_dev = diagnostic_dev.loc[diagnostic_dev.Hotel.eq(hotel)]
    for fold_number, fold_spec in enumerate(split_config["folds"], start=1):
        fit_date = pd.Timestamp(fold_spec["fit_date"])
        train = hotel_dev.loc[
            hotel_dev.CutoffDate.lt(fit_date)
            & hotel_dev.LabelAvailableDate.lt(fit_date)
            & hotel_dev.ExpectedCheckoutDate.lt(fit_date)
        ]
        val = hotel_dev.loc[hotel_dev.ValFold.eq(fold_number)]
        dummy = diagnostic_predictions[(hotel, "Dummy")].loc[
            lambda frame: frame.fold.eq(fold_number)
        ]
        rf = diagnostic_predictions[(hotel, "RF")].loc[
            lambda frame: frame.fold.eq(fold_number)
        ]
        assert len(val) == len(dummy) == len(rf)
        assert len(train) == diagnostic_metrics[(hotel, "RF")].loc[
            lambda frame: frame.fold.eq(fold_number) & frame.split.eq("train"),
            "n_bookings",
        ].iloc[0]
        val_rate = val.IsCanceled.mean()
        rf_mean = rf.p_cancel.mean()
        fold_rate_rows.append({
            "Hotel": hotel, "Fold": f"F{fold_number}",
            "Train N": len(train), "Train Cancel Rate": train.IsCanceled.mean(),
            "Val N": len(val), "Val Cancel Rate": val_rate,
            "Dummy Avg P": dummy.p_cancel.mean(), "RF Avg P": rf_mean,
            "RF Bias (pp)": 100 * (rf_mean - val_rate),
        })
        # Each displayed month contains only outcomes mature by this fold's
        # FitDate. The last six eligible cutoff months keep the trend compact.
        monthly = train.assign(
            CutoffMonth=train.CutoffDate.dt.to_period("M").dt.to_timestamp()
        ).groupby("CutoffMonth", as_index=False).agg(
            N=("Booking_ID", "size"), CancelRate=("IsCanceled", "mean")
        ).tail(6)
        for row in monthly.itertuples(index=False):
            history_rows.append({
                "Hotel": hotel, "As-of Fold": f"F{fold_number}",
                "FitDate": fit_date, "Cutoff Month": row.CutoffMonth,
                "Eligible N": row.N, "Cancel Rate": row.CancelRate,
            })

fold_rates = pd.DataFrame(fold_rate_rows)
historical_rates = pd.DataFrame(history_rows)
for hotel in ("H1", "H2"):
    print(f"{hotel} — fold-level rates (rates are fractions; bias is pp)")
    display(fold_rates.loc[fold_rates.Hotel.eq(hotel)].drop(columns="Hotel").round(4))
    print(f"{hotel} — last six cutoff months eligible as of each FitDate")
    history = historical_rates.loc[historical_rates.Hotel.eq(hotel)]
    display(history.drop(columns="Hotel").round(4))
    fig, ax = plt.subplots(figsize=(9, 3.5))
    for fold_name, series in history.groupby("As-of Fold", sort=True):
        ax.plot(series["Cutoff Month"], series["Cancel Rate"], marker="o", label=fold_name)
    ax.set(title=f"{hotel} — completed historical cutoff cohorts", xlabel="Cutoff month",
           ylabel="Cancellation rate")
    ax.legend(ncol=4)
    fig.autofmt_xdate()
    display(fig)
    plt.close(fig)

# Fold 1 composition gives a direct cohort-mix comparison alongside rates.
h2 = diagnostic_dev.loc[diagnostic_dev.Hotel.eq("H2")]
fit_date = pd.Timestamp(split_config["folds"][0]["fit_date"])
h2_train_f1 = h2.loc[
    h2.CutoffDate.lt(fit_date) & h2.LabelAvailableDate.lt(fit_date)
    & h2.ExpectedCheckoutDate.lt(fit_date)
]
h2_val_f1 = h2.loc[h2.ValFold.eq(1)]
composition = []
for name, cohort in (("Train F1", h2_train_f1), ("Val F1", h2_val_f1)):
    counts = cohort.groupby("MarketSegment", dropna=False).agg(
        N=("Booking_ID", "size"), CancelRate=("IsCanceled", "mean")
    ).reset_index()
    counts["Cohort"] = name
    counts["Share %"] = 100 * counts.N / len(cohort)
    composition.append(counts)
print("H2 Fold 1 — MarketSegment composition (same cohorts as the rate table)")
display(pd.concat(composition, ignore_index=True).round(3))

## 3.2 Probability Error Analysis

**Question:** Is RF overprediction spread across folds and risk buckets, particularly H2's 20–40% predicted-risk range?

In [ ]:
# Fixed 10-point buckets, with p=1 included in the final bucket. Empty
# buckets remain in the table with N=0 and undefined rates/gaps as NaN.
bucket_labels = [f"{10 * i}–{10 * (i + 1)}%" for i in range(10)]
probability_rows = []
for hotel in ("H1", "H2"):
    rf = diagnostic_predictions[(hotel, "RF")].copy()
    rf["bucket_id"] = np.minimum((rf.p_cancel * 10).astype(int), 9)
    grouped = rf.groupby(["fold", "bucket_id"]).agg(
        N=("Booking_ID", "size"), PredSum=("p_cancel", "sum"),
        ActualSum=("y_true", "sum"),
    )
    complete_index = pd.MultiIndex.from_product(
        [range(1, 5), range(10)], names=["fold", "bucket_id"]
    )
    grouped = grouped.reindex(complete_index, fill_value=0).reset_index()
    grouped["Avg Pred P"] = grouped.PredSum.div(grouped.N.replace(0, np.nan))
    grouped["Actual Rate"] = grouped.ActualSum.div(grouped.N.replace(0, np.nan))
    grouped["Gap (pp)"] = 100 * (grouped["Avg Pred P"] - grouped["Actual Rate"])
    grouped["Excess Predicted"] = grouped.PredSum - grouped.ActualSum
    grouped["Hotel"] = hotel
    grouped["Fold"] = grouped.fold.map(lambda number: f"F{number}")
    grouped["Risk Bucket"] = grouped.bucket_id.map(dict(enumerate(bucket_labels)))
    probability_rows.append(grouped)

probability_buckets = pd.concat(probability_rows, ignore_index=True)
columns = ["Fold", "Risk Bucket", "N", "Avg Pred P", "Actual Rate",
           "Gap (pp)", "Excess Predicted"]
for hotel in ("H1", "H2"):
    table = probability_buckets.loc[probability_buckets.Hotel.eq(hotel)]
    print(f"{hotel} — RF fold-level reliability; N=0 has undefined rates")
    display(table[columns].rename(columns={"N": "N Bookings"}).round(3))
    if hotel == "H2":
        print("H2 — RF 20–40% buckets across Validation folds")
        display(table.loc[
            table.bucket_id.isin([2, 3]), columns
        ].rename(columns={"N": "N Bookings"}).round(3))
    fig, axes = plt.subplots(2, 2, figsize=(9, 8), sharex=True, sharey=True)
    for fold_number, ax in enumerate(axes.flat, start=1):
        fold_rows = table.loc[
            table.fold.eq(fold_number) & table.N.gt(0)
        ]
        ax.plot([0, 1], [0, 1], "k--", linewidth=1)
        ax.scatter(fold_rows["Avg Pred P"], fold_rows["Actual Rate"],
                   s=20 + 3 * np.sqrt(fold_rows.N), alpha=0.75)
        ax.set(title=f"F{fold_number}", xlim=(0, 1), ylim=(0, 1))
    fig.suptitle(f"{hotel} — RF reliability by Validation fold (marker size ∝ N)")
    fig.supxlabel("Mean predicted probability")
    fig.supylabel("Actual cancellation rate")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

## 3.3 Segment Error Analysis

**Question:** Which booking segments account for RF prediction gaps, and do their gaps recur across Validation folds?

In [ ]:
# One-to-one Gold join; all RF rows are previously validated OOF bookings.
rf_parts = [diagnostic_predictions[(hotel, "RF")] for hotel in ("H1", "H2")]
rf_oof = pd.concat(rf_parts, ignore_index=True)
segment_source = diagnostic_dev[
    keys + ["MarketSegment", "CustomerType", "LeadTime", "ArrivalDateMonth"]
]
assert not segment_source.duplicated(keys).any()
rf_segments = rf_oof.merge(segment_source, on=keys, validate="one_to_one")
assert len(rf_segments) == len(rf_oof)

# LeadTime intervals are closed on the right: 0–30, 31–90, 91–180,
# 181–365 and 366+ days. Missing categorical values stay visible.
lead_labels = ["0–30", "31–90", "91–180", "181–365", "366+"]
rf_segments["LeadTime bucket"] = pd.cut(
    rf_segments.LeadTime,
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=lead_labels,
).astype("object").fillna("Missing")
dimensions = {
    "MarketSegment": "MarketSegment",
    "CustomerType": "CustomerType",
    "LeadTime bucket (days)": "LeadTime bucket",
    "ArrivalDateMonth": "ArrivalDateMonth",
}
for hotel in ("H1", "H2"):
    hotel_rows = rf_segments.loc[rf_segments.Hotel.eq(hotel)]
    for dimension, column in dimensions.items():
        source = hotel_rows.assign(Segment=hotel_rows[column].fillna("Missing"))
        grouped = source.groupby(["fold", "Segment"], observed=True).agg(
            N=("Booking_ID", "size"), PredSum=("p_cancel", "sum"),
            ActualSum=("y_true", "sum"),
        ).reset_index()
        grouped["Actual Rate"] = grouped.ActualSum / grouped.N
        grouped["RF Avg P"] = grouped.PredSum / grouped.N
        grouped["Gap (pp)"] = 100 * (grouped["RF Avg P"] - grouped["Actual Rate"])
        grouped["Excess Predicted"] = grouped.PredSum - grouped.ActualSum
        grouped["Fold"] = grouped.fold.map(lambda number: f"F{number}")
        detail = grouped[["Fold", "Segment", "N", "Actual Rate", "RF Avg P",
                          "Gap (pp)", "Excess Predicted"]].rename(
            columns={"N": "N Bookings"}
        )
        totals = grouped.groupby("Segment").agg(
            **{"Total N": ("N", "sum"),
               "Total Excess Predicted": ("Excess Predicted", "sum")}
        )
        gaps = grouped.pivot(index="Segment", columns="fold", values="Gap (pp)")
        gaps = gaps.reindex(columns=range(1, 5))
        gaps.columns = [f"F{fold} Gap (pp)" for fold in range(1, 5)]
        across_folds = totals.join(gaps).reset_index()
        across_folds = across_folds.sort_values(
            "Total Excess Predicted", key=lambda values: values.abs(),
            ascending=False,
        )
        print(f"{hotel} — {dimension}: segment errors by fold")
        display(detail.round(3))
        print(f"{hotel} — {dimension}: persistence and total contribution")
        display(across_folds.round(3))

## 3.4 Daily Error Analysis

**Question:** Are RF's daily cancellation errors driven by bias, a few large-error dates, or booking-volume and calendar patterns compared with Dummy?

In [ ]:
# Match the same OOF bookings in RF and Dummy; StayDate is ArrivalDate.
daily_frames = {}
daily_summaries = []
top_days = {}
for hotel in ("H1", "H2"):
    rf = diagnostic_predictions[(hotel, "RF")][
        keys + ["fold", "StayDate", "y_true", "p_cancel"]
    ].rename(columns={"p_cancel": "RF P"})
    dummy = diagnostic_predictions[(hotel, "Dummy")][
        keys + ["fold", "StayDate", "y_true", "p_cancel"]
    ].rename(columns={"p_cancel": "Dummy P"})
    matched = rf.merge(
        dummy, on=keys + ["fold"], validate="one_to_one",
        suffixes=("", "_dummy"),
    )
    assert len(matched) == len(rf) == len(dummy)
    assert matched.StayDate.eq(matched.StayDate_dummy).all()
    assert matched.y_true.eq(matched.y_true_dummy).all()
    daily = matched.groupby(["fold", "StayDate"], as_index=False).agg(
        **{"N Bookings": ("Booking_ID", "size"),
           "Actual Cancellations": ("y_true", "sum"),
           "RF Expected": ("RF P", "sum"),
           "Dummy Expected": ("Dummy P", "sum")}
    )
    daily["RF Error"] = daily["RF Expected"] - daily["Actual Cancellations"]
    daily["Dummy Error"] = daily["Dummy Expected"] - daily["Actual Cancellations"]
    daily["RF Absolute Error"] = daily["RF Error"].abs()
    daily["Dummy Absolute Error"] = daily["Dummy Error"].abs()
    daily["RF Squared Error"] = daily["RF Error"].pow(2)
    daily["Dummy Squared Error"] = daily["Dummy Error"].pow(2)
    daily["Fold"] = daily.fold.map(lambda number: f"F{number}")
    daily["Arrival Weekday"] = daily.StayDate.dt.day_name()
    daily["Arrival Month"] = daily.StayDate.dt.month_name()
    daily_frames[hotel] = daily

    for fold_number, fold_days in daily.groupby("fold"):
        mean_actual = fold_days["Actual Cancellations"].mean()
        rf_rmse = np.sqrt(fold_days["RF Squared Error"].mean())
        dummy_rmse = np.sqrt(fold_days["Dummy Squared Error"].mean())
        rf_nrmse = rf_rmse / mean_actual
        dummy_nrmse = dummy_rmse / mean_actual
        for model, computed in (("RF", rf_nrmse), ("Dummy", dummy_nrmse)):
            saved = diagnostic_metrics[(hotel, model)].loc[
                lambda frame: frame.fold.eq(fold_number) & frame.split.eq("val"),
                "daily_nrmse",
            ].iloc[0]
            assert np.isclose(computed, saved)
        daily_summaries.append({
            "Hotel": hotel, "Fold": f"F{fold_number}",
            "N Days": len(fold_days),
            "Mean RF Daily Error": fold_days["RF Error"].mean(),
            "% Days Overpredicted": 100 * fold_days["RF Error"].gt(0).mean(),
            "RF Daily RMSE": rf_rmse, "Dummy Daily RMSE": dummy_rmse,
            "RF Daily nRMSE": rf_nrmse, "Dummy Daily nRMSE": dummy_nrmse,
        })

    # Top-day contribution denominator is this hotel's total RF squared
    # error across every fold-day, with fold and ArrivalDate kept distinct.
    denominator = daily["RF Squared Error"].sum()
    largest = daily.nlargest(15, "RF Absolute Error").copy()
    largest["% Total Squared Error"] = (
        100 * largest["RF Squared Error"] / denominator
    )
    top_days[hotel] = largest

daily_summary = pd.DataFrame(daily_summaries)
for hotel in ("H1", "H2"):
    daily = daily_frames[hotel]
    print(f"{hotel} — daily OOF errors by ArrivalDate (scroll to inspect all days)")
    shown = daily[["Fold", "StayDate", "N Bookings", "Actual Cancellations",
                   "RF Expected", "Dummy Expected", "RF Error", "Dummy Error"]]
    html = shown.to_html(index=False, float_format=lambda value: f"{value:.3f}")
    display(HTML(f'<div style="max-height:320px;overflow:auto">{html}</div>'))
    print(f"{hotel} — fold-level daily metrics")
    display(daily_summary.loc[daily_summary.Hotel.eq(hotel)].drop(columns="Hotel").round(3))
    print(f"{hotel} — 15 largest RF daily errors; denominator = {daily['RF Squared Error'].sum():.3f} total RF squared error across all Validation fold-days")
    largest = top_days[hotel][
        ["Fold", "StayDate", "N Bookings", "Actual Cancellations", "RF Expected",
         "RF Error", "RF Absolute Error", "% Total Squared Error",
         "Arrival Weekday", "Arrival Month"]
    ]
    display(largest.round(3))

    # Summarize all fold-days by volume and calendar, without treating
    # descriptive associations as causes of an error.
    assoc = daily.copy()
    assoc["Volume Quartile"] = pd.qcut(
        assoc["N Bookings"], q=4, duplicates="drop"
    ).astype(str)
    association_parts = []
    for dimension in ("Volume Quartile", "Arrival Weekday", "Arrival Month"):
        grouped = assoc.groupby(dimension, observed=True).agg(
            **{"N Days": ("StayDate", "size"),
               "Mean Bookings": ("N Bookings", "mean"),
               "Mean Abs RF Error": ("RF Absolute Error", "mean"),
               "Mean RF Error": ("RF Error", "mean")}
        ).reset_index().rename(columns={dimension: "Group"})
        grouped.insert(0, "Dimension", dimension)
        association_parts.append(grouped)
    print(f"{hotel} — daily errors by volume and arrival calendar")
    display(pd.concat(association_parts, ignore_index=True).round(3))

# Composition on the largest-error dates: dominant MarketSegment share.
for hotel in ("H2", "H1"):
    rf = diagnostic_predictions[(hotel, "RF")][keys + ["fold", "StayDate"]]
    segments = rf.merge(
        diagnostic_dev[keys + ["MarketSegment"]], on=keys,
        validate="one_to_one",
    )
    counts = segments.groupby(
        ["fold", "StayDate", "MarketSegment"], dropna=False
    ).size().rename("Segment N").reset_index()
    dominant = counts.sort_values(
        ["fold", "StayDate", "Segment N"], ascending=[True, True, False]
    ).drop_duplicates(["fold", "StayDate"])
    composition = top_days[hotel].merge(
        dominant, on=["fold", "StayDate"], validate="one_to_one"
    )
    composition["Dominant Segment Share %"] = (
        100 * composition["Segment N"] / composition["N Bookings"]
    )
    print(f"{hotel} — booking composition on the 15 largest-error days")
    display(composition[["Fold", "StayDate", "N Bookings", "MarketSegment",
                         "Dominant Segment Share %"]].round(2))

    # Three focused panels use Fold 1 for a consistent cross-hotel view.
    fold_days = daily_frames[hotel].loc[lambda frame: frame.fold.eq(1)]
    fig, axes = plt.subplots(3, 1, figsize=(11, 9))
    axes[0].plot(fold_days.StayDate, fold_days["Actual Cancellations"], label="Actual", color="black")
    axes[0].plot(fold_days.StayDate, fold_days["RF Expected"], label="RF")
    axes[0].plot(fold_days.StayDate, fold_days["Dummy Expected"], label="Dummy")
    axes[0].set(ylabel="Cancellations", title=f"{hotel} Fold 1 — daily actual vs expected")
    axes[0].legend(ncol=3)
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].plot(fold_days.StayDate, fold_days["RF Error"])
    axes[1].set(ylabel="RF expected − actual", title="RF daily residual")
    axes[2].scatter(fold_days["N Bookings"], fold_days["RF Absolute Error"], alpha=0.7)
    axes[2].set(xlabel="Scored bookings on arrival date", ylabel="Absolute RF error",
                title="RF daily error magnitude vs booking volume")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

## 3.5 Model Sanity & Generalization Checks

**Question:** Do saved Train–Validation gaps and extreme incorrect probabilities help locate LR, RF, or XGB anomalies, especially H2 LR Fold 2?

In [ ]:
# Pair Train and Validation Log Loss from the same saved fold. No model is
# refitted, and the probability checks use OOF Validation rows only.
metrics_all = pd.concat(diagnostic_metrics.values(), ignore_index=True)
generalization = metrics_all.pivot(
    index=["hotel", "model_name", "fold"], columns="split", values="log_loss"
).reset_index()
generalization["Gap"] = generalization["val"] - generalization["train"]
generalization = generalization.rename(columns={
    "hotel": "Hotel", "model_name": "Model", "fold": "Fold",
    "train": "Train Log Loss", "val": "Val Log Loss",
})
generalization["Fold"] = generalization.Fold.map(lambda number: f"F{number}")
for hotel in ("H1", "H2"):
    print(f"{hotel} — saved fold-level Train–Validation Log Loss")
    display(generalization.loc[generalization.Hotel.eq(hotel)].round(4))

# Thresholds are inclusive: near-zero p <= 0.01; near-one p >= 0.99.
# Wrong extremes are p <= 0.01 with y=1 or p >= 0.99 with y=0.
extreme_rows = []
for hotel in ("H1", "H2"):
    for model in ("LR", "RF", "XGB"):
        saved = diagnostic_predictions[(hotel, model)]
        for fold_number, scored in saved.groupby("fold"):
            p = scored.p_cancel.to_numpy()
            y = scored.y_true.to_numpy()
            p_safe = np.clip(p, np.finfo(float).eps, 1 - np.finfo(float).eps)
            row_loss = -(y * np.log(p_safe) + (1 - y) * np.log1p(-p_safe))
            wrong_extreme = ((p <= 0.01) & (y == 1)) | ((p >= 0.99) & (y == 0))
            score = diagnostic_metrics[(hotel, model)].loc[
                lambda frame: frame.fold.eq(fold_number) & frame.split.eq("val"),
                "log_loss",
            ].iloc[0]
            assert np.isclose(row_loss.mean(), score)
            extreme_rows.append({
                "Hotel": hotel, "Model": model, "Fold": f"F{fold_number}",
                "N": len(scored), "Min P": p.min(), "Median P": np.median(p),
                "Max P": p.max(), "Near-0 Count": int((p <= 0.01).sum()),
                "Near-1 Count": int((p >= 0.99).sum()),
                "Wrong Extreme N": int(wrong_extreme.sum()),
                "Wrong Extreme Loss Share %": 100 * row_loss[wrong_extreme].sum() / row_loss.sum(),
                "Val Log Loss": score,
            })
probability_extremes = pd.DataFrame(extreme_rows)
print("Probability thresholds: near-zero <= 0.01; near-one >= 0.99; wrong extreme contradicts the label")
for hotel in ("H1", "H2"):
    print(f"{hotel} — OOF probability distribution and extremes")
    display(probability_extremes.loc[probability_extremes.Hotel.eq(hotel)].round(4))

# Inspect the individual H2 LR Fold 2 rows with the largest saved loss.
anomalous = diagnostic_predictions[("H2", "LR")].loc[
    lambda frame: frame.fold.eq(2)
].copy()
p_safe = np.clip(anomalous.p_cancel.to_numpy(),
                 np.finfo(float).eps, 1 - np.finfo(float).eps)
y = anomalous.y_true.to_numpy()
anomalous["Row Log Loss"] = -(y * np.log(p_safe) + (1 - y) * np.log1p(-p_safe))
print("H2 LR Fold 2 — 10 highest individual OOF Log Loss contributions")
display(anomalous.nlargest(10, "Row Log Loss")[
    ["Booking_ID", "y_true", "p_cancel", "Row Log Loss"]
].round(4))

# Estimator parameters are persisted. Fitted preprocessing and convergence
# diagnostics are absent from current experiment artifacts, so say so.
parameter_keys = {
    "LR": ("C", "penalty", "solver", "max_iter", "class_weight"),
    "RF": ("n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight"),
    "XGB": ("n_estimators", "learning_rate", "max_depth", "subsample", "colsample_bytree"),
}
config_rows = []
for hotel in ("H1", "H2"):
    for model in ("LR", "RF", "XGB"):
        saved = diagnostic_configs[(hotel, model)]
        params = saved["estimator_params"]
        config_rows.append({
            "Hotel": hotel, "Model": model,
            "Saved Estimator": saved["estimator"],
            "Saved Parameters": ", ".join(
                f"{name}={params[name]}" for name in parameter_keys[model]
                if name in params
            ),
            "Preprocessing Settings": saved.get("preprocessing", "Not saved in run artifact"),
            "Convergence Diagnostics": saved.get("convergence_diagnostics", "Not saved in run artifact"),
        })
print("Saved model configuration and diagnostic availability")
display(pd.DataFrame(config_rows))

## Diagnostic Analysis — Key Insights & Next Actions

- **Temporal Drift:** H2 RF substantially overpredicts in Fold 1 but underpredicts in Fold 2. → **Action:** Test recency weighting to evaluate whether emphasizing recent training data improves temporal stability.

- **Probability Reliability:** RF overprediction is concentrated in certain risk buckets and varies across folds, particularly for H2. → **Action:** Defer global calibration until model improvements are evaluated; reassess probability reliability afterward.

- **Segment Errors:** Several H2 market segments show persistent prediction bias across folds. → **Action:** Test segment-based feature engineering, particularly LeadTime interactions and historical segment cancellation rates.

- **Daily Forecasting:** H2 daily errors are driven by both systematic bias and a few extreme-error days, some dominated by group bookings. → **Action:** Investigate group-related booking patterns and test targeted feature interactions.

- **Generalization & Model Sanity:** RF/XGB show noticeable Train–Validation gaps, while LR produces extreme incorrect probabilities in certain folds. → **Action:** Test regularization for tree models and investigate LR configuration before further model comparison.

# 4. Experiment Screening

## 4.1 Training Strategy — Recency Weighting

Test RF30 with three independent time-decay weights because Section 4 showed fold-dependent RF probability bias, especially for H2.

In [ ]:
# Read only Gold Development rows and saved split_v2 Development membership.
# Baseline folders are read-only; each new intervention has a unique _s1 ID.
import matplotlib.pyplot as plt
from utils.model_screening import run_screening_experiment

screening_membership = ds.dataset(
    split_dir / "membership.parquet", format="parquet"
).to_table(filter=ds.field("Partition") == "Dev").to_pandas()
screening_model_data = pd.concat([
    ds.dataset(
        gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet",
        format="parquet",
    ).to_table(
        filter=ds.field("CutoffDate") < pd.Timestamp(split_config["test_start"]).date()
    ).to_pandas()
    for hotel in ("H1", "H2")
], ignore_index=True)
screening_model_data["CutoffDate"] = pd.to_datetime(screening_model_data.CutoffDate)
assert screening_membership.Partition.eq("Dev").all()
assert screening_model_data.CutoffDate.lt(pd.Timestamp(split_config["test_start"])).all()
assert not screening_membership.duplicated(keys).any()
assert not screening_model_data.duplicated(keys).any()
assert len(feature_columns) == 30
screening_source_fingerprint = json.loads(
    (split_dir / "config.json").read_text(encoding="utf-8")
)["source_fingerprint"]

screening_results = {}
screening_variants = []
screening_status = []

def load_screening_baseline(hotel, model):
    '''Read and align an existing 30-feature baseline without refitting.'''
    run_id = f"{hotel}_{model}_v2"
    folder = experiment_root / run_id
    config = json.loads((folder / "config.json").read_text(encoding="utf-8"))
    metrics = pd.read_parquet(folder / "metrics.parquet")
    predictions = pd.read_parquet(folder / "predictions.parquet")
    assert config["split_id"] == "split_v2"
    assert config["feature_count"] == 30
    assert config["selected_features"] == selected_features
    expected = screening_membership.loc[
        screening_membership.Hotel.eq(hotel) & screening_membership.ValFold.notna(),
        keys + ["ValFold", "ArrivalDate"],
    ].merge(
        screening_model_data.loc[
            screening_model_data.Hotel.eq(hotel), keys + ["IsCanceled"]
        ], on=keys, validate="one_to_one",
    )
    aligned = predictions.merge(expected, on=keys, validate="one_to_one")
    assert len(aligned) == len(predictions) == len(expected)
    assert not predictions.duplicated(keys).any()
    assert aligned.fold.eq(aligned.ValFold).all()
    assert aligned.y_true.eq(aligned.IsCanceled).all()
    assert aligned.StayDate.eq(aligned.ArrivalDate).all()
    return {"config": config, "metrics": metrics, "predictions": predictions,
            "status": "baseline loaded and verified"}

for hotel in ("H1", "H2"):
    for model in ("RF", "XGB", "LR"):
        screening_results[(hotel, f"{model}30")] = load_screening_baseline(hotel, model)

def run_screening_variant(hotel, label, track, baseline, spec):
    '''Run/load one independent variant and record its comparison identity.'''
    run_id = f"{hotel}_{label}_s1"
    result = run_screening_experiment(
        experiment_id=run_id, hotel=hotel, spec=spec,
        model_data=screening_model_data,
        membership=screening_membership,
        selected_features=selected_features,
        feature_contract=feature_contract,
        split_config=split_config,
        output_root=experiment_root,
        source_fingerprint=screening_source_fingerprint,
    )
    screening_results[(hotel, label)] = result
    screening_variants.append({
        "Hotel": hotel, "Track": track, "Experiment": label,
        "Baseline": baseline,
    })
    screening_status.append({
        "Hotel": hotel, "Track": track, "Experiment": label,
        "Status": result["status"],
    })
    return result

def fold_scores(result, column, split="val"):
    '''Return one saved metric per fold in chronological fold order.'''
    frame = result["metrics"]
    return frame.loc[frame.split.eq(split)].sort_values("fold")[column].to_numpy()

def mean_sd(result, column):
    '''Display equal-fold mean and sample SD without pooling OOF rows.'''
    values = fold_scores(result, column)
    return f"{np.mean(values):.4f} ± {np.std(values, ddof=1):.4f}"

for hotel in ("H1", "H2"):
    for half_life in (90, 180, 365):
        run_screening_variant(
            hotel, f"RF_RW{half_life}", "Training Strategy", "RF30",
            {
                "track": "Recency Weighting", "family": "RF",
                "estimator_params": model_parameters["RF"],
                "half_life_days": half_life, "seed": 42,
            },
        )

rows, fold_rows, weight_rows, rate_rows = [], [], [], []
for hotel in ("H1", "H2"):
    baseline = screening_results[(hotel, "RF30")]
    for label in ("RF30", "RF_RW90", "RF_RW180", "RF_RW365"):
        result = screening_results[(hotel, label)]
        rows.append({
            "Hotel": hotel,
            "Configuration": "RF30 Baseline" if label == "RF30" else label.removeprefix("RF_"),
            "Val Log Loss": mean_sd(result, "log_loss"),
            "Daily nRMSE": mean_sd(result, "daily_nrmse"),
            "Daily Bias %": mean_sd(result, "daily_bias_pct"),
            "ROC-AUC": mean_sd(result, "roc_auc"),
        })
        losses = fold_scores(result, "log_loss")
        base_losses = fold_scores(baseline, "log_loss")
        fold_rows.append({
            "Hotel": hotel, "Configuration": label,
            **{f"F{index} Log Loss": value for index, value in enumerate(losses, 1)},
            "Mean Δ vs Baseline": float(np.mean(losses - base_losses))
            if label != "RF30" else np.nan,
        })
        for fold_number in range(1, 5):
            scored = result["predictions"].loc[
                lambda frame: frame.fold.eq(fold_number)
            ]
            rate_rows.append({
                "Hotel": hotel, "Fold": f"F{fold_number}",
                "Configuration": label, "Val N": len(scored),
                "Actual Rate": scored.y_true.mean(),
                "Avg Pred P": scored.p_cancel.mean(),
            })
            if label == "RF30":
                train_n = int(fold_scores(baseline, "n_bookings", "train")[fold_number - 1])
                weight_rows.append({
                    "Hotel": hotel, "Fold": f"F{fold_number}",
                    "Configuration": label, "Train N": train_n,
                    "Min Weight": 1.0, "Max Weight": 1.0,
                    "Effective Sample Size": train_n,
                })
            else:
                detail = result["config"]["fold_details"][fold_number - 1]["weights"]
                assert np.isclose(detail["mean_weight"], 1.0)
                assert result["config"]["fold_details"][fold_number - 1]["fit_detail"]["weighted_estimator_fit"]
                weight_rows.append({
                    "Hotel": hotel, "Fold": f"F{fold_number}",
                    "Configuration": label, "Train N": detail["train_n"],
                    "Min Weight": detail["min_weight"],
                    "Max Weight": detail["max_weight"],
                    "Effective Sample Size": detail["effective_sample_size"],
                })
print("5.1 — four-fold performance (mean ± sample SD)")
display(pd.DataFrame(rows))
print("5.1 — paired fold Log Loss; negative Δ indicates lower Log Loss")
display(pd.DataFrame(fold_rows).round(4))
print("5.1 — actual Validation rate and mean predicted probability")
display(pd.DataFrame(rate_rows).round(4))
print("5.1 — normalized training-weight distribution and ESS")
display(pd.DataFrame(weight_rows).round(3))

## 4.2 Feature Engineering Experiments

Test four separate additions to RF30, motivated by Section 4's H2 segment errors and group-related daily errors; each keeps the original 30 predictors.

In [ ]:
# FE1 uses train segment medians; FE2 train-defined segment interactions;
# FE3 fixed Groups interactions; FE4 strictly prior completed outcomes.
for hotel in ("H1", "H2"):
    for feature_experiment in ("FE1", "FE2", "FE3", "FE4"):
        run_screening_variant(
            hotel, f"RF_{feature_experiment}", "Feature Engineering", "RF30",
            {
                "track": "Feature Engineering", "family": "RF",
                "estimator_params": model_parameters["RF"],
                "feature_experiment": feature_experiment,
                "seed": 42,
            },
        )

performance, fold_table = [], []
for hotel in ("H1", "H2"):
    baseline = screening_results[(hotel, "RF30")]
    for label in ("RF30", "RF_FE1", "RF_FE2", "RF_FE3", "RF_FE4"):
        result = screening_results[(hotel, label)]
        if label == "RF30":
            added_counts = [0] * 4
        else:
            added_counts = [
                len(detail["added_features"])
                for detail in result["config"]["fold_details"]
            ]
        added_display = (
            str(added_counts[0]) if len(set(added_counts)) == 1
            else f"{min(added_counts)}–{max(added_counts)} by fold"
        )
        total_display = (
            str(30 + added_counts[0]) if len(set(added_counts)) == 1
            else f"{30 + min(added_counts)}–{30 + max(added_counts)}"
        )
        losses = fold_scores(result, "log_loss")
        base_losses = fold_scores(baseline, "log_loss")
        performance.append({
            "Hotel": hotel, "Experiment": label,
            "Added Features": added_display, "Total Features": total_display,
            "Val Log Loss": mean_sd(result, "log_loss"),
            "Δ vs RF30": float(np.mean(losses - base_losses)) if label != "RF30" else np.nan,
            "Daily nRMSE": mean_sd(result, "daily_nrmse"),
            "ROC-AUC": mean_sd(result, "roc_auc"),
        })
        fold_table.append({
            "Hotel": hotel, "Experiment": label,
            **{f"F{index} Log Loss": value for index, value in enumerate(losses, 1)},
            "Mean Δ vs RF30": float(np.mean(losses - base_losses))
            if label != "RF30" else np.nan,
        })
print("5.2 — independent feature experiments (four-fold mean ± sample SD)")
display(pd.DataFrame(performance).round(4))
print("5.2 — paired fold Log Loss")
display(pd.DataFrame(fold_table).round(4))

# Same H2 OOF bookings for every variant; segment gaps are percentage points.
target_segments = ["Offline TA/TO", "Direct", "Corporate", "Groups", "Online TA"]
segment_rows = []
h2_segments = screening_model_data.loc[
    screening_model_data.Hotel.eq("H2"), keys + ["MarketSegment"]
]
for label in ("RF30", "RF_FE1", "RF_FE2", "RF_FE3", "RF_FE4"):
    scored = screening_results[("H2", label)]["predictions"].merge(
        h2_segments, on=keys, validate="one_to_one"
    )
    for segment in target_segments:
        rows = scored.loc[scored.MarketSegment.eq(segment)]
        if rows.empty:
            continue
        segment_rows.append({
            "Experiment": label, "Segment": segment, "N": len(rows),
            "Actual Rate": rows.y_true.mean(), "Avg Pred P": rows.p_cancel.mean(),
            "Gap (pp)": 100 * (rows.p_cancel.mean() - rows.y_true.mean()),
            "Excess Predicted": float((rows.p_cancel - rows.y_true).sum()),
        })
print("5.2 — targeted H2 MarketSegment OOF comparison")
display(pd.DataFrame(segment_rows).round(3))

## 4.3 Random Forest Regularization

Test a compact RF grid around the saved max depth and leaf size because Section 4 showed Train–Validation Log Loss gaps.

In [ ]:
# Baseline is max_depth=None, min_samples_leaf=20. All other parameters,
# Gold predictors, uniform weights and fold membership stay fixed.
rf_grid = [
    ("RF_T1", None, 40),
    ("RF_T2", None, 80),
    ("RF_T3", 16, 20),
    ("RF_T4", 16, 40),
    ("RF_T5", 12, 40),
]
for hotel in ("H1", "H2"):
    for label, max_depth, min_leaf in rf_grid:
        params = dict(model_parameters["RF"])
        params.update(max_depth=max_depth, min_samples_leaf=min_leaf)
        run_screening_variant(
            hotel, label, "RF Regularization", "RF30",
            {"track": "RF Regularization", "family": "RF",
             "estimator_params": params, "seed": 42},
        )

configurations, performance, stability = [], [], []
for hotel in ("H1", "H2"):
    baseline = screening_results[(hotel, "RF30")]
    for label in ("RF30",) + tuple(item[0] for item in rf_grid):
        result = screening_results[(hotel, label)]
        params = (model_parameters["RF"] if label == "RF30"
                  else result["config"]["request"]["spec"]["estimator_params"])
        train_loss = fold_scores(result, "log_loss", "train")
        val_loss = fold_scores(result, "log_loss")
        configurations.append({
            "Hotel": hotel, "Configuration": label,
            "Max Depth": str(params["max_depth"]),
            "Min Samples Leaf": params["min_samples_leaf"],
            "Val Log Loss": float(val_loss.mean()),
            "Train Log Loss": float(train_loss.mean()),
            "Gap": float((val_loss - train_loss).mean()),
        })
        performance.append({
            "Hotel": hotel, "Configuration": label,
            "Val Log Loss": mean_sd(result, "log_loss"),
            "Daily nRMSE": mean_sd(result, "daily_nrmse"),
            "Daily Bias %": mean_sd(result, "daily_bias_pct"),
            "ROC-AUC": mean_sd(result, "roc_auc"),
            "Δ Log Loss": float((val_loss - fold_scores(baseline, "log_loss")).mean())
            if label != "RF30" else np.nan,
        })
        stability.append({
            "Hotel": hotel, "Configuration": label,
            **{f"F{index}": value for index, value in enumerate(val_loss, 1)},
            "Mean": float(val_loss.mean()), "SD": float(val_loss.std(ddof=1)),
        })
print("5.3 — saved RF parameters and Train–Validation gap (four-fold means)")
display(pd.DataFrame(configurations).round(4))
print("5.3 — full performance (four-fold mean ± sample SD)")
display(pd.DataFrame(performance).round(4))
print("5.3 — fold-level Validation Log Loss")
display(pd.DataFrame(stability).round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, hotel in zip(axes, ("H1", "H2")):
    frame = pd.DataFrame(configurations).loc[lambda data: data.Hotel.eq(hotel)]
    ax.plot(frame.Configuration, frame["Train Log Loss"], marker="o", label="Train")
    ax.plot(frame.Configuration, frame["Val Log Loss"], marker="o", label="Validation")
    ax.set(title=hotel, ylabel="Mean Log Loss")
    ax.tick_params(axis="x", rotation=40)
    ax.legend()
fig.tight_layout()
display(fig)
plt.close(fig)

## 4.4 Model Alternatives

Compare LightGBM, CatBoost and official TabM with RF30 and XGB30 on the unchanged 30-feature cohorts, as Section 4 identified model-specific probability behavior.

In [ ]:
# One documented initial configuration per family; no feature engineering,
# weighting, calibration or outer-Validation early stopping is used.
alternative_specs = {
    "LightGBM": {
        "track": "Model Alternatives", "family": "LightGBM", "seed": 42,
        "estimator_params": {
            "objective": "binary", "n_estimators": 200,
            "learning_rate": 0.05, "num_leaves": 15, "max_depth": 6,
            "min_child_samples": 40, "random_state": 42,
            "n_jobs": 2, "verbosity": -1, "deterministic": True,
            "force_col_wise": True,
        },
    },
    "CatBoost": {
        "track": "Model Alternatives", "family": "CatBoost", "seed": 42,
        "estimator_params": {
            "iterations": 300, "learning_rate": 0.05, "depth": 5,
            "l2_leaf_reg": 5, "loss_function": "Logloss",
            "random_seed": 42, "thread_count": 2,
            "allow_writing_files": False, "verbose": False,
        },
    },
    "TabM": {
        "track": "Model Alternatives", "family": "TabM", "seed": 42,
        "architecture": {"d_block": 64, "n_blocks": 2, "k": 8, "dropout": 0.1},
        "training_params": {
            "optimizer": "AdamW", "learning_rate": 0.001,
            "weight_decay": 0.0001, "batch_size": 1024, "epochs": 8,
        },
    },
}
for hotel in ("H1", "H2"):
    for family in ("LightGBM", "CatBoost", "TabM"):
        try:
            run_screening_variant(
                hotel, family, "Model Alternatives", "RF30",
                alternative_specs[family],
            )
        except (ImportError, ModuleNotFoundError, RuntimeError) as error:
            screening_status.append({
                "Hotel": hotel, "Track": "Model Alternatives",
                "Experiment": family, "Status": f"unavailable: {type(error).__name__}: {error}",
            })

comparison, fold_table, reliability_rows, config_rows = [], [], [], []
for hotel in ("H1", "H2"):
    for model in ("RF30", "XGB30", "LightGBM", "CatBoost", "TabM"):
        if (hotel, model) not in screening_results:
            continue
        result = screening_results[(hotel, model)]
        train = fold_scores(result, "log_loss", "train")
        val = fold_scores(result, "log_loss")
        comparison.append({
            "Hotel": hotel, "Model": model,
            "Val Log Loss": mean_sd(result, "log_loss"),
            "Daily nRMSE": mean_sd(result, "daily_nrmse"),
            "Daily Bias %": mean_sd(result, "daily_bias_pct"),
            "ROC-AUC": mean_sd(result, "roc_auc"),
            "Train–Val Gap": float((val - train).mean()),
        })
        fold_table.append({
            "Hotel": hotel, "Model": model,
            **{f"F{index} Log Loss": value for index, value in enumerate(val, 1)},
            "Mean": float(val.mean()), "SD": float(val.std(ddof=1)),
        })
        scored = result["predictions"].copy()
        scored["bucket"] = np.minimum((scored.p_cancel * 10).astype(int), 9)
        grouped = scored.groupby("bucket").agg(
            N=("Booking_ID", "size"), PredSum=("p_cancel", "sum"),
            ActualSum=("y_true", "sum"),
        ).reindex(range(10), fill_value=0)
        for bucket, row in grouped.iterrows():
            reliability_rows.append({
                "Hotel": hotel, "Model": model,
                "Probability Bucket": f"{bucket * 10}–{(bucket + 1) * 10}%",
                "N Bookings": int(row.N),
                "Mean Predicted P": row.PredSum / row.N if row.N else np.nan,
                "Actual Rate": row.ActualSum / row.N if row.N else np.nan,
            })
        if model in alternative_specs:
            details = result["config"]["fold_details"]
            first = details[0]["fit_detail"]
            config_rows.append({
                "Hotel": hotel, "Model": model,
                "Key Hyperparameters": str(alternative_specs[model].get(
                    "estimator_params", alternative_specs[model].get("architecture")
                )),
                "Preprocessing": first["preprocessing"],
                "Training Method": (
                    "member-level BCE; fixed epochs" if model == "TabM"
                    else "four independent outer-fold fits"
                ),
                "Device": first.get("device", "CPU"),
                "Mean Training Seconds": np.mean([
                    detail["fit_detail"]["training_seconds"] for detail in details
                ]),
            })
print("5.4 — model-family comparison (four-fold mean ± sample SD)")
display(pd.DataFrame(comparison).round(4))
print("5.4 — fold-level Validation Log Loss")
display(pd.DataFrame(fold_table).round(4))
for hotel in ("H1", "H2"):
    print(f"5.4 — {hotel} pooled OOF reliability; separate from fold-averaged metrics")
    display(pd.DataFrame(reliability_rows).loc[
        lambda frame: frame.Hotel.eq(hotel)
    ].round(4))
print("5.4 — saved model configuration and training method")
display(pd.DataFrame(config_rows).round(2))
print("5.4 — execution status")
display(pd.DataFrame(screening_status).loc[
    lambda frame: frame.Track.eq("Model Alternatives")
])

## 4.5 Logistic Regression Configuration

Test stronger L2 regularization and a separate iteration-budget change because Section 4 showed unstable LR probabilities, especially H2 Fold 2.

In [ ]:
# The saved LR baseline uses numeric median+scaling, categorical one-hot,
# lbfgs, C=1 and max_iter=1000. Each variant changes one parameter only.
lr_variants = [
    ("LR_C0p3", {"C": 0.3}),
    ("LR_C0p1", {"C": 0.1}),
    ("LR_Iter2000", {"max_iter": 2000}),
]
for hotel in ("H1", "H2"):
    for label, change in lr_variants:
        params = dict(model_parameters["LR"])
        params.update(change)
        run_screening_variant(
            hotel, label, "LR Configuration", "LR30",
            {"track": "LR Configuration", "family": "LR",
             "estimator_params": params, "seed": 42},
        )

configurations, fold_rows, extremes = [], [], []
for hotel in ("H1", "H2"):
    for label in ("LR30",) + tuple(item[0] for item in lr_variants):
        result = screening_results[(hotel, label)]
        params = (model_parameters["LR"] if label == "LR30"
                  else result["config"]["request"]["spec"]["estimator_params"])
        if label == "LR30":
            convergence = "not saved in baseline artifact"
        else:
            warnings_by_fold = [
                len(detail["fit_detail"]["convergence_warnings"])
                for detail in result["config"]["fold_details"]
            ]
            convergence = f"{sum(warnings_by_fold)} warnings across 4 folds"
        configurations.append({
            "Hotel": hotel, "Configuration": label,
            "Solver": params["solver"], "C": params["C"],
            "Max Iter": params["max_iter"], "Convergence Status": convergence,
            "Val Log Loss": mean_sd(result, "log_loss"),
            "ROC-AUC": mean_sd(result, "roc_auc"),
        })
        losses = fold_scores(result, "log_loss")
        fold_rows.append({
            "Hotel": hotel, "Configuration": label,
            **{f"F{index} Log Loss": value for index, value in enumerate(losses, 1)},
            "Mean": float(losses.mean()),
        })
        for fold_number in range(1, 5):
            scored = result["predictions"].loc[
                lambda frame: frame.fold.eq(fold_number)
            ]
            p = scored.p_cancel
            y = scored.y_true
            extremes.append({
                "Hotel": hotel, "Configuration": label,
                "Fold": f"F{fold_number}", "N": len(scored),
                "Near-0 Count": int(p.le(0.01).sum()),
                "Near-1 Count": int(p.ge(0.99).sum()),
                "Extreme Wrong Count": int((
                    (p.le(0.01) & y.eq(1)) | (p.ge(0.99) & y.eq(0))
                ).sum()),
                "Val Log Loss": float(losses[fold_number - 1]),
            })
print("5.5 — LR configuration; thresholds near-0 <=0.01, near-1 >=0.99")
display(pd.DataFrame(configurations))
print("5.5 — fold-level Validation Log Loss")
display(pd.DataFrame(fold_rows).round(4))
print("5.5 — extreme OOF probability counts")
display(pd.DataFrame(extremes).round(4))

## 4.6 Screening Comparison

Compare each completed intervention with its own saved RF30 or LR30 baseline using paired folds, without combining changes or selecting a final configuration.

In [ ]:
# Every row compares the same hotel's four saved ValFold cohorts. Mean
# differences are computed from paired fold scores, not pooled OOF rows.
master_rows, fold_delta_rows = [], []
for item in screening_variants:
    hotel, label, baseline_name = item["Hotel"], item["Experiment"], item["Baseline"]
    result = screening_results[(hotel, label)]
    baseline = screening_results[(hotel, baseline_name)]
    val = fold_scores(result, "log_loss")
    base = fold_scores(baseline, "log_loss")
    delta = val - base
    train = fold_scores(result, "log_loss", "train")
    master_rows.append({
        "Hotel": hotel, "Track": item["Track"], "Experiment": label,
        "Baseline": baseline_name,
        "Val Log Loss": float(val.mean()),
        "Val Log Loss SD": float(val.std(ddof=1)),
        "Δ vs Baseline": float(delta.mean()),
        "Δ SD": float(delta.std(ddof=1)),
        "Folds Improved": f"{int((delta < 0).sum())}/4",
        "Daily nRMSE": float(fold_scores(result, "daily_nrmse").mean()),
        "Daily nRMSE SD": float(fold_scores(result, "daily_nrmse").std(ddof=1)),
        "Daily Bias %": float(fold_scores(result, "daily_bias_pct").mean()),
        "Daily Bias SD": float(fold_scores(result, "daily_bias_pct").std(ddof=1)),
        "ROC-AUC": float(fold_scores(result, "roc_auc").mean()),
        "ROC-AUC SD": float(fold_scores(result, "roc_auc").std(ddof=1)),
        "Train–Val Gap": float((val - train).mean()),
        "Train–Val Gap SD": float((val - train).std(ddof=1)),
    })
    fold_delta_rows.append({
        "Hotel": hotel, "Track": item["Track"], "Experiment": label,
        **{f"F{index} Δ": value for index, value in enumerate(delta, 1)},
        "Mean Δ": float(delta.mean()),
    })
master = pd.DataFrame(master_rows)
print("5.6 — independent screening results; four-fold mean and sample SD")
for hotel in ("H1", "H2"):
    print(hotel)
    display(master.loc[master.Hotel.eq(hotel)].round(4))
print("5.6 — paired fold Log Loss change (negative means lower loss)")
for hotel in ("H1", "H2"):
    print(hotel)
    display(pd.DataFrame(fold_delta_rows).loc[
        lambda frame: frame.Hotel.eq(hotel)
    ].round(4))
tradeoff = master[["Hotel", "Track", "Experiment", "Folds Improved", "Δ vs Baseline"]].copy()
tradeoff["Δ Daily nRMSE"] = [
    fold_scores(screening_results[(item["Hotel"], item["Experiment"])], "daily_nrmse").mean()
    - fold_scores(screening_results[(item["Hotel"], item["Baseline"])], "daily_nrmse").mean()
    for item in screening_variants
]
print("5.6 — Log Loss / daily nRMSE trade-off by hotel")
for hotel in ("H1", "H2"):
    print(hotel)
    display(tradeoff.loc[tradeoff.Hotel.eq(hotel)].round(4))
if screening_status:
    print("5.6 — experiment execution status, including unavailable runs")
    display(pd.DataFrame(screening_status))

### Controlled Experiment Screening — Key Insights & Decisions

**H1 — Resort Hotel**

- **Recency Weighting & Feature Engineering:** RW365 and FE3 slightly improve Log Loss but not Daily nRMSE. → **Decision:** Keep RW365 as an optional improvement candidate; do not prioritize further feature engineering.
- **RF Regularization:** Reducing model complexity does not improve validation performance. → **Decision:** Retain the original RF30 configuration.
- **Model Alternatives:** CatBoost performs comparably to RF30 with substantially lower prediction bias, while XGB remains competitive. → **Decision:** Retain CatBoost and XGB as potential ensemble candidates.
- **Logistic Regression:** Stronger regularization improves LR performance, but it remains weaker than RF. → **Decision:** Do not prioritize further LR optimization.

**H2 — City Hotel**

- **Recency Weighting:** RW90 achieves the lowest Validation Log Loss, while RW180 improves Log Loss across all four folds. → **Decision:** Retain both for combined experiments.
- **Feature Engineering:** FE4 (Historical Segment Cancellation Rate) improves Log Loss across all four folds, outperforming other engineered feature sets. → **Decision:** Retain FE4 for combination testing.
- **RF Regularization:** Reduced model complexity improves Validation Log Loss and narrows the generalization gap. → **Decision:** Retain RF-T4 and RF-T5 for combination testing.
- **Model Alternatives:** TabM achieves the best Daily nRMSE, outperforming both RF30 and Dummy, but has slightly worse Log Loss than RF30. LightGBM and CatBoost do not outperform RF30. → **Decision:** Retain TabM as an independent challenger and examine its training convergence; deprioritize LightGBM and CatBoost.

**Next Steps**

- **H1:** Keep RF30 as the primary candidate; test RW365 as a limited improvement and retain CatBoost/XGB for potential ensemble experiments.
- **H2:** Run controlled combinations of recency weighting, FE4, and RF regularization to test whether their individual improvements are complementary.
- **TabM:** Validate training convergence and stability before deciding whether to retain it for ensemble evaluation.
- **Model Selection:** Compare combined configurations against RF30 and the best standalone experiments before proceeding to Ensemble and Calibration.

# 5. Combined Improvement Experiments

## 5.1 H1 — Limited Combination Experiments

Test whether RW365 and FE3 complement one another on H1, using the saved RF30 and standalone results as matched references.

In [ ]:
# Section 6 uses Development-only Gold rows and the exact saved split_v2
# membership. No earlier training cell must rerun to reproduce these outputs.
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import log_loss, roc_auc_score
from utils.model_screening import run_screening_experiment

combined_membership = ds.dataset(
    split_dir / "membership.parquet", format="parquet"
).to_table(filter=ds.field("Partition") == "Dev").to_pandas()
combined_data = pd.concat([
    ds.dataset(
        gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet",
        format="parquet",
    ).to_table(
        filter=ds.field("CutoffDate") < pd.Timestamp(split_config["test_start"]).date()
    ).to_pandas()
    for hotel in ("H1", "H2")
], ignore_index=True)
combined_data["CutoffDate"] = pd.to_datetime(combined_data.CutoffDate)
assert combined_membership.Partition.eq("Dev").all()
assert combined_data.CutoffDate.lt(pd.Timestamp(split_config["test_start"])).all()
assert not combined_membership.duplicated(keys).any()
assert not combined_data.duplicated(keys).any()
assert len(feature_columns) == 30
combined_source_fingerprint = json.loads(
    (split_dir / "config.json").read_text(encoding="utf-8")
)["source_fingerprint"]
combined_results = {}
combined_status = []

def display_label(label):
    """Name the saved eight-epoch TabM baseline consistently in tables."""
    if label == "TabM":
        return "TabM-E8"
    return label.replace("TabM_E", "TabM-E")

def load_combined_reference(hotel, label, expected_spec=None):
    """Verify one immutable baseline or screening artifact before comparison."""
    run_id = f"{hotel}_{'RF_v2' if label == 'RF30' else label + '_s1'}"
    folder = experiment_root / run_id
    assert folder.exists(), f"Required standalone artifact is missing: {run_id}"
    if expected_spec is not None:
        result = run_screening_experiment(
            experiment_id=run_id, hotel=hotel, spec=expected_spec,
            model_data=combined_data, membership=combined_membership,
            selected_features=selected_features,
            feature_contract=feature_contract, split_config=split_config,
            output_root=experiment_root,
            source_fingerprint=combined_source_fingerprint,
        )
    else:
        config = json.loads((folder / "config.json").read_text(encoding="utf-8"))
        assert config["split_id"] == "split_v2"
        assert config["feature_count"] == 30
        assert config["selected_features"] == selected_features
        metrics = pd.read_parquet(folder / "metrics.parquet")
        predictions = pd.read_parquet(folder / "predictions.parquet")
        expected = combined_membership.loc[
            combined_membership.Hotel.eq(hotel) & combined_membership.ValFold.notna(),
            keys + ["ValFold", "CutoffDate", "ArrivalDate"],
        ].merge(
            combined_data.loc[
                combined_data.Hotel.eq(hotel), keys + ["IsCanceled"]
            ], on=keys, validate="one_to_one",
        )
        aligned = predictions.merge(
            expected, on=keys, validate="one_to_one", suffixes=("", "_expected")
        )
        assert len(aligned) == len(predictions) == len(expected)
        assert not predictions.duplicated(keys).any()
        assert aligned.fold.eq(aligned.ValFold).all()
        assert aligned.y_true.eq(aligned.IsCanceled).all()
        assert aligned.CutoffDate.eq(aligned.CutoffDate_expected).all()
        assert aligned.StayDate.eq(aligned.ArrivalDate).all()
        for fold_number in range(1, 5):
            scored = predictions.loc[predictions.fold.eq(fold_number)]
            metric = metrics.loc[
                metrics.fold.eq(fold_number) & metrics.split.eq("val")
            ].iloc[0]
            assert metric.n_bookings == len(scored)
            assert np.isclose(metric.log_loss, log_loss(
                scored.y_true, scored.p_cancel, labels=[0, 1]
            ))
            assert np.isclose(metric.roc_auc, roc_auc_score(
                scored.y_true, scored.p_cancel
            ))
        result = {"config": config, "metrics": metrics,
                  "predictions": predictions, "status": "loaded and verified"}
    combined_results[(hotel, label)] = result
    combined_status.append({"Hotel": hotel, "Experiment": display_label(label),
                            "Status": result["status"], "Artifact ID": run_id})
    return result

def run_combination(hotel, label, spec):
    """Create or verify one versioned combination using the Section 5 runner."""
    run_id = f"{hotel}_{label}_c1"
    result = run_screening_experiment(
        experiment_id=run_id, hotel=hotel, spec=spec,
        model_data=combined_data, membership=combined_membership,
        selected_features=selected_features,
        feature_contract=feature_contract, split_config=split_config,
        output_root=experiment_root,
        source_fingerprint=combined_source_fingerprint,
    )
    combined_results[(hotel, label)] = result
    combined_status.append({"Hotel": hotel, "Experiment": display_label(label),
                            "Status": result["status"], "Artifact ID": run_id})
    return result

def values(hotel, label, metric, split="val"):
    """Read four chronological fold metrics; never pool OOF rows here."""
    frame = combined_results[(hotel, label)]["metrics"]
    selected = frame.loc[frame.split.eq(split)].sort_values("fold")
    assert selected.fold.tolist() == [1, 2, 3, 4]
    return selected[metric].to_numpy(dtype=float)

def mean_sd(hotel, label, metric, split="val"):
    """Format equal-fold mean and sample standard deviation."""
    scores = values(hotel, label, metric, split)
    return f"{scores.mean():.4f} ± {scores.std(ddof=1):.4f}"

def performance_table(hotel, labels):
    """Summarize four-fold Validation metrics against this hotel's RF30."""
    baseline = values(hotel, "RF30", "log_loss")
    return pd.DataFrame([{
        "Configuration": display_label(label),
        "Val Log Loss": mean_sd(hotel, label, "log_loss"),
        "Δ vs RF30": float(np.mean(values(hotel, label, "log_loss") - baseline)),
        "Daily nRMSE": mean_sd(hotel, label, "daily_nrmse"),
        "Daily Bias %": mean_sd(hotel, label, "daily_bias_pct"),
        "ROC-AUC": mean_sd(hotel, label, "roc_auc"),
    } for label in labels])

def fold_table(hotel, labels, metric="log_loss"):
    """Keep paired fold values and baseline-improvement counts visible."""
    baseline = values(hotel, "RF30", metric)
    return pd.DataFrame([{
        "Configuration": display_label(label),
        **{f"F{fold}": score for fold, score in enumerate(
            values(hotel, label, metric), 1
        )},
        "Mean": float(values(hotel, label, metric).mean()),
        "SD": float(values(hotel, label, metric).std(ddof=1)),
        "Folds Improved vs RF30": int(np.sum(
            values(hotel, label, metric) < baseline
        )) if label != "RF30" else pd.NA,
    } for label in labels])

rf_params = model_parameters["RF"].copy()
assert rf_params["max_depth"] is None and rf_params["min_samples_leaf"] == 20
load_combined_reference("H1", "RF30")
load_combined_reference("H1", "RF_RW365", {
    "track": "Recency Weighting", "family": "RF",
    "estimator_params": rf_params, "half_life_days": 365, "seed": 42,
})
load_combined_reference("H1", "RF_FE3", {
    "track": "Feature Engineering", "family": "RF",
    "estimator_params": rf_params, "feature_experiment": "FE3", "seed": 42,
})
run_combination("H1", "C1_RW365_FE3", {
    "track": "Combined RF", "family": "RF",
    "estimator_params": rf_params, "half_life_days": 365,
    "feature_experiment": "FE3", "seed": 42,
})
h1_labels = ["RF30", "RF_RW365", "RF_FE3", "C1_RW365_FE3"]
print("H1 — Performance (mean ± sample SD across four Val folds)")
display(performance_table("H1", h1_labels).round(4))
print("H1 — Fold-level Validation Log Loss")
display(fold_table("H1", h1_labels).round(4))

## 5.2 H2 — Combined RF Experiments

Test four predefined two-intervention combinations using the saved H2 standalones, with FE4 history, recency weights, and RF parameters unchanged from screening.

In [ ]:
# T4 and T5 change only depth and leaf size relative to the saved RF30.
t4_params = {**rf_params, "max_depth": 16, "min_samples_leaf": 40}
t5_params = {**rf_params, "max_depth": 12, "min_samples_leaf": 40}
load_combined_reference("H2", "RF30")
for half_life in (90, 180):
    load_combined_reference("H2", f"RF_RW{half_life}", {
        "track": "Recency Weighting", "family": "RF",
        "estimator_params": rf_params,
        "half_life_days": half_life, "seed": 42,
    })
load_combined_reference("H2", "RF_FE4", {
    "track": "Feature Engineering", "family": "RF",
    "estimator_params": rf_params,
    "feature_experiment": "FE4", "seed": 42,
})
for label, params in (("RF_T4", t4_params), ("RF_T5", t5_params)):
    load_combined_reference("H2", label, {
        "track": "RF Regularization", "family": "RF",
        "estimator_params": params, "seed": 42,
    })

h2_combinations = {
    "C1_RW90_FE4": (rf_params, 90, "FE4", ("RF_RW90", "RF_FE4")),
    "C2_RW180_FE4": (rf_params, 180, "FE4", ("RF_RW180", "RF_FE4")),
    "C3_T4_FE4": (t4_params, None, "FE4", ("RF_T4", "RF_FE4")),
    "C4_T5_RW90": (t5_params, 90, None, ("RF_T5", "RF_RW90")),
}
for label, (params, half_life, feature, _) in h2_combinations.items():
    spec = {
        "track": "Combined RF", "family": "RF",
        "estimator_params": params, "seed": 42,
    }
    if half_life is not None:
        spec["half_life_days"] = half_life
    if feature is not None:
        spec["feature_experiment"] = feature
    run_combination("H2", label, spec)

h2_labels = ["RF30", *h2_combinations]
print("H2 — Combination performance (mean ± sample SD)")
display(performance_table("H2", h2_labels).round(4))
print("H2 — Fold-level Validation Log Loss")
display(fold_table("H2", h2_labels).round(4))

# Compare paired fold scores against the better constituent in each fold,
# and separately compare equal-fold means against the better standalone mean.
standalone_rows, paired_rows = [], []
base = values("H2", "RF30", "log_loss")
for label, (*_, constituents) in h2_combinations.items():
    component_scores = np.vstack([
        values("H2", item, "log_loss") for item in constituents
    ])
    combined_scores = values("H2", label, "log_loss")
    best_fold = component_scores.min(axis=0)
    best_mean = component_scores.mean(axis=1).min()
    standalone_rows.append({
        "Combination": label, "Constituents": ", ".join(constituents),
        "RF30 Log Loss": base.mean(),
        "Best Constituent Log Loss": best_mean,
        "Combined Log Loss": combined_scores.mean(),
        "Δ vs Best Constituent": combined_scores.mean() - best_mean,
    })
    paired_rows.append({
        "Combination": label,
        **{f"F{fold} Δ vs best constituent": value for fold, value in
           enumerate(combined_scores - best_fold, 1)},
    })
print("H2 — Combination vs standalone (equal-fold means)")
display(pd.DataFrame(standalone_rows).round(4))
print("H2 — Paired fold differences vs better constituent in each fold")
display(pd.DataFrame(paired_rows).round(4))

print("H2 — Generalization (unweighted Train and Val Log Loss)")
display(pd.DataFrame([{
    "Configuration": label,
    "Train Log Loss": mean_sd("H2", label, "log_loss", "train"),
    "Val Log Loss": mean_sd("H2", label, "log_loss"),
    "Gap": float(np.mean(
        values("H2", label, "log_loss")
        - values("H2", label, "log_loss", "train")
    )),
} for label in h2_labels]).round(4))

# All four folds retain the same validation bookings. F1 is shown first.
probability_rows = []
for label in h2_labels:
    predictions = combined_results[("H2", label)]["predictions"]
    for fold in range(1, 5):
        scored = predictions.loc[predictions.fold.eq(fold)]
        probability_rows.append({
            "Configuration": label, "Fold": f"F{fold}", "Val N": len(scored),
            "Actual Rate": scored.y_true.mean(),
            "Avg Pred P": scored.p_cancel.mean(),
            "Probability Gap (pp)": 100 * (
                scored.p_cancel.mean() - scored.y_true.mean()
            ),
            "Daily nRMSE": values("H2", label, "daily_nrmse")[fold - 1],
            "Daily Bias %": values("H2", label, "daily_bias_pct")[fold - 1],
        })
print("H2 — Probability and daily fit by Val fold (F1 first)")
display(pd.DataFrame(probability_rows).sort_values(
    ["Fold", "Configuration"]
).round(4))

## 5.3 TabM — Training Convergence Check

Compare the saved eight-epoch TabM with fixed 20- and 40-epoch budgets on H2, changing only training duration and using no outer-fold early stopping.

In [ ]:
tabm_base = {
    "track": "Model Alternatives", "family": "TabM", "seed": 42,
    "architecture": {"d_block": 64, "n_blocks": 2, "k": 8, "dropout": 0.1},
    "training_params": {
        "optimizer": "AdamW", "learning_rate": 0.001,
        "weight_decay": 0.0001, "batch_size": 1024, "epochs": 8,
    },
}
load_combined_reference("H2", "TabM", tabm_base)
for epochs in (20, 40):
    tabm_spec = json.loads(json.dumps(tabm_base))
    tabm_spec["track"] = "TabM Training Budget"
    tabm_spec["training_params"]["epochs"] = epochs
    run_combination("H2", f"TabM_E{epochs}", tabm_spec)

tabm_labels = ["RF30", "TabM", "TabM_E20", "TabM_E40"]
behavior_rows = []
for label in tabm_labels[1:]:
    details = combined_results[("H2", label)]["config"]["fold_details"]
    for item in details:
        fit = item["fit_detail"]
        losses = fit["train_loss_by_epoch"]
        behavior_rows.append({
            "Hotel": "H2", "Configuration": f"TabM-E{len(losses)}",
            "Fold": f"F{item['fold']}", "Epoch Budget": len(losses),
            "Actual Epochs": len(losses), "Final Train Loss": losses[-1],
            "Internal Holdout Loss": pd.NA,
            "Training Time (s)": fit["training_seconds"],
        })
print("H2 — TabM training behavior (fixed budgets; no internal holdout)")
display(pd.DataFrame(behavior_rows).round(4))
fig, ax = plt.subplots(figsize=(8, 4))
for label in tabm_labels[1:]:
    details = combined_results[("H2", label)]["config"]["fold_details"]
    curves = np.array([item["fit_detail"]["train_loss_by_epoch"]
                       for item in details])
    ax.plot(np.arange(1, curves.shape[1] + 1), curves.mean(axis=0),
            marker="o", markersize=2, label=f"TabM-E{curves.shape[1]}")
ax.set(xlabel="Epoch", ylabel="Mean member-level Train BCE",
       title="H2 TabM training loss across four folds")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
display(fig)
plt.close(fig)

print("H2 — TabM performance (mean ± sample SD)")
display(performance_table("H2", tabm_labels).assign(
    **{"Train–Val Gap": [float(np.mean(
        values("H2", label, "log_loss")
        - values("H2", label, "log_loss", "train")
    )) for label in tabm_labels]}
).round(4))
print("H2 — Fold-level Validation Log Loss")
display(fold_table("H2", tabm_labels).round(4))
print("H2 — Fold-level Daily nRMSE")
display(fold_table("H2", tabm_labels, "daily_nrmse").round(4))

## 5.4 Combined Experiment Comparison

Place H1 and H2 combined configurations beside their saved references, using paired fold differences for Log Loss and separate daily-fit comparisons.

In [ ]:
comparison = {
    "H1": ["RF30", "RF_RW365", "RF_FE3", "C1_RW365_FE3"],
    "H2": [
        "RF30", "RF_RW90", "RF_RW180", "RF_FE4", "RF_T4", "RF_T5",
        *h2_combinations, "TabM", "TabM_E20", "TabM_E40",
    ],
}
master_rows, fold_delta_rows, tradeoff_rows = [], [], []
for hotel, labels in comparison.items():
    base_loss = values(hotel, "RF30", "log_loss")
    base_daily = values(hotel, "RF30", "daily_nrmse")
    for label in labels:
        losses = values(hotel, label, "log_loss")
        daily = values(hotel, label, "daily_nrmse")
        gap = np.mean(losses - values(hotel, label, "log_loss", "train"))
        best_component_delta = np.nan
        if hotel == "H2" and label in h2_combinations:
            components = h2_combinations[label][-1]
            best_component_delta = losses.mean() - min(
                values(hotel, item, "log_loss").mean()
                for item in components
            )
        elif hotel == "H1" and label == "C1_RW365_FE3":
            best_component_delta = losses.mean() - min(
                values(hotel, item, "log_loss").mean()
                for item in ("RF_RW365", "RF_FE3")
            )
        master_rows.append({
            "Hotel": hotel, "Configuration": display_label(label),
            "Val Log Loss": mean_sd(hotel, label, "log_loss"),
            "Δ vs RF30": (losses - base_loss).mean(),
            "Folds Improved": int((losses < base_loss).sum())
            if label != "RF30" else pd.NA,
            "Daily nRMSE": mean_sd(hotel, label, "daily_nrmse"),
            "Daily Bias %": mean_sd(hotel, label, "daily_bias_pct"),
            "ROC-AUC": mean_sd(hotel, label, "roc_auc"),
            "Train–Val Gap": gap,
            "Δ vs Best Standalone": best_component_delta,
        })
        fold_delta_rows.append({
            "Hotel": hotel, "Configuration": display_label(label),
            **{f"F{fold} Δ": difference for fold, difference in
               enumerate(losses - base_loss, 1)},
            "Mean Δ": (losses - base_loss).mean(),
        })
        tradeoff_rows.append({
            "Hotel": hotel, "Configuration": display_label(label),
            "Δ Val Log Loss": (losses - base_loss).mean(),
            "Δ Daily nRMSE": (daily - base_daily).mean(),
            "Folds Improved": int((losses < base_loss).sum())
            if label != "RF30" else pd.NA,
        })

print("Combined comparison — Four-fold means ± sample SD")
display(pd.DataFrame(master_rows).round(4))
print("Combined comparison — Paired Val Log Loss difference from RF30")
display(pd.DataFrame(fold_delta_rows).round(4))
print("Combined comparison — Performance trade-off")
display(pd.DataFrame(tradeoff_rows).round(4))
for hotel in ("H1", "H2"):
    points = pd.DataFrame(tradeoff_rows).loc[
        lambda frame: frame.Hotel.eq(hotel)
    ].sort_values("Δ Val Log Loss")
    positions = np.arange(len(points))
    fig, axes = plt.subplots(
        1, 2, figsize=(10, max(3.2, 0.4 * len(points) + 1)),
        sharey=True,
    )
    for ax, column, title in zip(
        axes,
        ("Δ Val Log Loss", "Δ Daily nRMSE"),
        ("Validation Log Loss", "Daily nRMSE"),
    ):
        ax.barh(positions, points[column], color="steelblue")
        ax.axvline(0, color="gray", linewidth=0.8)
        ax.set(xlabel=f"Δ vs RF30 — {title}", title=title)
        ax.grid(axis="x", alpha=0.2)
    axes[0].set_yticks(positions, points.Configuration)
    axes[0].invert_yaxis()
    fig.suptitle(f"{hotel} — Combined experiment trade-off")
    plt.tight_layout()
    display(fig)
    plt.close(fig)
print("Experiment status and immutable artifact IDs")
display(pd.DataFrame(combined_status))

### Combined Improvement Experiments — Key Insights & Decisions

**H1 — Resort Hotel**

- **Combined Improvements:** RW365 + FE3 does not outperform standalone RW365 on Log Loss and worsens Daily nRMSE. → **Decision:** Discard the combined configuration.
- **Model Performance:** RF30 remains the strongest baseline for daily forecasting, while RW365 achieves a marginally lower Log Loss (0.2612 vs 0.2622). → **Decision:** Retain RF30 and RW365 as single-model candidates.
- **Further Improvement:** No substantial benefit from additional feature combinations or RF regularization. → **Decision:** Stop further RF improvement experiments for H1.

**H2 — City Hotel**

- **Combined Improvements:** All four combinations improve mean Validation Log Loss and Daily nRMSE over RF30, indicating complementary benefits from selected interventions.
- **Best Mean Performance:** C4 (RF-T5 + RW90) achieves the lowest Log Loss (0.3457) and Daily nRMSE (0.9543), but improves Log Loss in only 2/4 folds. → **Decision:** Retain C4 as the performance-oriented candidate.
- **Temporal Stability:** C2 (RW180 + FE4) improves Log Loss across all four folds, achieving 0.3462 mean Log Loss. → **Decision:** Retain C2 as the stability-oriented candidate.
- **Remaining Limitations:** Combined models still exhibit substantial overprediction in H2 Fold 1, and C4's Daily nRMSE remains slightly worse than Dummy. → **Decision:** Reassess probability reliability after model selection.

**TabM — Training Convergence**

- **Training Duration:** Increasing from 8 to 20–40 epochs worsens Validation Log Loss and Daily nRMSE despite decreasing training loss, indicating overfitting.
- **Model Performance:** TabM-E8 retains the best observed H2 Daily nRMSE (0.8904), but its Log Loss (0.3530) remains worse than RF30. → **Decision:** Retain TabM-E8 as an independent challenger; discard E20/E40 and stop further TabM tuning for now.

**Decision: Ensemble Evaluation**

- **H1:** Evaluate ensemble eligibility among RF30, RW365, CatBoost, and XGB30; test probability averaging only for models showing complementary errors.
- **H2:** Evaluate ensemble eligibility among C2, C4, and TabM-E8, using RF30 as the baseline; test whether ensemble predictions improve Log Loss, Daily nRMSE, and temporal stability over the best single model.

# 6. Ensemble Evaluation

## 6.1 Ensemble Eligibility Analysis

Assess prediction similarity and error complementarity among the shortlisted models after verifying that every model scores the identical Validation bookings, comparing booking-level probability errors and daily cancellation forecast errors separately for each hotel.

In [ ]:
# Section 7 reads saved Development OOF predictions and split_v2 membership
# only. No base model is refitted and no Test outcome is loaded.
import json
from itertools import combinations

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from utils.model_ensemble import align_components, run_ensemble, verify_component

membership_dataset = ds.dataset(split_dir / "membership.parquet", format="parquet")
ensemble_membership = membership_dataset.to_table(
    filter=ds.field("Partition") == "Dev"
).to_pandas()
ensemble_test_keys = membership_dataset.to_table(
    columns=keys, filter=ds.field("Partition") == "Test"
).to_pandas()
ensemble_labels = pd.concat([
    ds.dataset(
        gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet",
        format="parquet",
    ).to_table(
        columns=keys + ["IsCanceled"],
        filter=ds.field("CutoffDate") < pd.Timestamp(split_config["test_start"]).date(),
    ).to_pandas()
    for hotel in ("H1", "H2")
], ignore_index=True)
assert ensemble_membership.Partition.eq("Dev").all()
assert not ensemble_membership.duplicated(keys).any()
assert not ensemble_labels.duplicated(keys).any()
assert ensemble_labels.IsCanceled.isin([0, 1]).all()
assert len(ensemble_membership[keys].merge(
    ensemble_labels, on=keys, validate="one_to_one"
)) == len(ensemble_membership) == len(ensemble_labels)
ensemble_source_fingerprint = json.loads(
    (split_dir / "config.json").read_text(encoding="utf-8")
)["source_fingerprint"]

def initial_identity(estimator, model):
    """Saved _v2 identity, encoded exactly as in Section 2."""
    return {
        "model_name": model,
        "experiment_version": development_config["experiment_version"],
        "estimator_params": {
            name: "NaN" if isinstance(value, float) and np.isnan(value) else value
            for name, value in estimator.get_params().items()
        },
    }

# Expected identities restate the Section 5–6 specs; any saved difference
# stops here instead of substituting another experiment.
rf_params = model_parameters["RF"]
shortlist = {
    "H1": {
        "RF30": ("H1_RF_v2", initial_identity(RandomForestClassifier(**rf_params), "RF")),
        "RW365": ("H1_RF_RW365_s1", {"spec": {
            "track": "Recency Weighting", "family": "RF",
            "estimator_params": rf_params, "half_life_days": 365, "seed": 42,
        }}),
        "CatBoost": ("H1_CatBoost_s1", {"spec": {
            "track": "Model Alternatives", "family": "CatBoost", "seed": 42,
            "estimator_params": {
                "iterations": 300, "learning_rate": 0.05, "depth": 5,
                "l2_leaf_reg": 5, "loss_function": "Logloss",
                "random_seed": 42, "thread_count": 2,
                "allow_writing_files": False, "verbose": False,
            },
        }}),
        "XGB30": ("H1_XGB_v2", initial_identity(
            XGBClassifier(**model_parameters["XGB"]), "XGB"
        )),
    },
    "H2": {
        "RF30": ("H2_RF_v2", initial_identity(RandomForestClassifier(**rf_params), "RF")),
        "C2": ("H2_C2_RW180_FE4_c1", {"spec": {
            "track": "Combined RF", "family": "RF", "estimator_params": rf_params,
            "seed": 42, "half_life_days": 180, "feature_experiment": "FE4",
        }}),
        "C4": ("H2_C4_T5_RW90_c1", {"spec": {
            "track": "Combined RF", "family": "RF",
            "estimator_params": {**rf_params, "max_depth": 12, "min_samples_leaf": 40},
            "seed": 42, "half_life_days": 90,
        }}),
        "TabM-E8": ("H2_TabM_s1", {"spec": {
            "track": "Model Alternatives", "family": "TabM", "seed": 42,
            "architecture": {"d_block": 64, "n_blocks": 2, "k": 8, "dropout": 0.1},
            "training_params": {
                "optimizer": "AdamW", "learning_rate": 0.001,
                "weight_decay": 0.0001, "batch_size": 1024, "epochs": 8,
            },
        }}),
    },
}

ensemble_components, ensemble_status = {}, []
ensemble_wide, ensemble_daily = {}, {}
component_rows, cohort_rows = [], []
for hotel, models in shortlist.items():
    for label, (artifact_id, identity) in models.items():
        component = verify_component(
            experiment_root=experiment_root, artifact_id=artifact_id,
            hotel=hotel, identity=identity, membership=ensemble_membership,
            labels=ensemble_labels, test_keys=ensemble_test_keys,
            selected_features=selected_features, split_config=split_config,
            source_fingerprint=ensemble_source_fingerprint,
        )
        ensemble_components[(hotel, label)] = component
        ensemble_status.append({"Hotel": hotel, "Experiment": label,
                                "Status": component["status"],
                                "Artifact ID": artifact_id})
        component_rows.append({
            "Hotel": hotel, "Model": label, "Artifact ID": artifact_id,
            "Validation Bookings": len(component["predictions"]),
            "Folds": component["predictions"].fold.nunique(),
            "Predictions SHA-256": component["fingerprint"]["predictions_sha256"][:12],
            "Status": component["status"],
        })

    # Explicit outer one-to-one joins: any unmatched booking, fold, label,
    # CutoffDate or StayDate difference raises instead of being dropped.
    wide = align_components({
        label: ensemble_components[(hotel, label)]["predictions"] for label in models
    })
    expected_n = int((
        ensemble_membership.Hotel.eq(hotel) & ensemble_membership.ValFold.notna()
    ).sum())
    assert len(wide) == expected_n and wide[list(models)].notna().all().all()
    assert not wide[keys].merge(ensemble_test_keys, on=keys).shape[0]
    ensemble_wide[hotel] = wide
    ensemble_daily[hotel] = wide.groupby(["fold", "StayDate"], as_index=False).agg(
        **{"N Bookings": ("Booking_ID", "size"),
           "Actual Cancellations": ("y_true", "sum")},
        **{label: (label, "sum") for label in models},
    )
    cohort_rows.append({
        "Hotel": hotel, "Models": len(models), "Matched Bookings": len(wide),
        **{f"F{fold} N": int(wide.fold.eq(fold).sum()) for fold in range(1, 5)},
        "Cancellations": int(wide.y_true.sum()),
        "Fold-Days": len(ensemble_daily[hotel]), "Test Keys": 0,
        "Identical": "keys, fold, label, CutoffDate, StayDate",
    })
print("7.1 — shortlisted artifacts (identity, OOF cohort and saved metrics re-verified)")
display(pd.DataFrame(component_rows))
print("7.1 — matched Validation cohorts used by every comparison")
display(pd.DataFrame(cohort_rows))

similarity_rows, error_rows, daily_rows = [], [], []
for hotel, models in shortlist.items():
    wide, daily = ensemble_wide[hotel], ensemble_daily[hotel]
    y = wide.y_true.to_numpy()
    for model_a, model_b in combinations(models, 2):
        p_a, p_b = wide[model_a].to_numpy(), wide[model_b].to_numpy()
        error_a, error_b = p_a - y, p_b - y
        gap = np.abs(p_a - p_b)
        pair = {"Hotel": hotel, "Model A": model_a, "Model B": model_b}
        similarity_rows.append({
            **pair,
            "Prediction Correlation": np.corrcoef(p_a, p_b)[0, 1],
            "Mean Absolute Disagreement": gap.mean(),
        })
        error_rows.append({
            **pair,
            "Signed Error Corr.": np.corrcoef(error_a, error_b)[0, 1],
            "Absolute Error Corr.": np.corrcoef(
                np.abs(error_a), np.abs(error_b)
            )[0, 1],
            "Mean Absolute Disagreement": gap.mean(),
            "Disagreement ≥0.10 (%)": 100 * np.mean(gap >= 0.10),
        })
        daily_a = daily[model_a] - daily["Actual Cancellations"]
        daily_b = daily[model_b] - daily["Actual Cancellations"]
        daily_rows.append({
            **pair, "Fold-Days": len(daily),
            "Daily Error Corr.": np.corrcoef(daily_a, daily_b)[0, 1],
            "Abs. Daily Error Corr.": np.corrcoef(daily_a.abs(), daily_b.abs())[0, 1],
            "Days A Better": int((daily_a.abs() < daily_b.abs()).sum()),
            "Days B Better": int((daily_b.abs() < daily_a.abs()).sum()),
            "Ties": int((daily_a.abs() == daily_b.abs()).sum()),
            "Mean Forecast Disagreement": float((daily[model_a] - daily[model_b]).abs().mean()),
        })
for title, rows in (
    ("pairwise prediction similarity (pooled OOF bookings, four folds)", similarity_rows),
    ("booking-level error complementarity (pooled OOF; disagreement threshold 0.10)", error_rows),
    ("daily error complementarity (ArrivalDate within ValFold; pooled fold-days)", daily_rows),
):
    for hotel in ("H1", "H2"):
        print(f"7.1 — {hotel} {title}")
        display(pd.DataFrame(rows).loc[lambda frame: frame.Hotel.eq(hotel)].round(4))

correlations = {
    hotel: ensemble_wide[hotel][list(models)].corr() for hotel, models in shortlist.items()
}
low = min(matrix.to_numpy().min() for matrix in correlations.values())
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), constrained_layout=True)
for ax, (hotel, matrix) in zip(axes, correlations.items()):
    image = ax.imshow(matrix, cmap="Blues", vmin=low, vmax=1)
    ax.set_xticks(range(len(matrix)), matrix.columns, rotation=30)
    ax.set_yticks(range(len(matrix)), matrix.index)
    for row in range(len(matrix)):
        for column in range(len(matrix)):
            value = matrix.iloc[row, column]
            ax.text(column, row, f"{value:.3f}", ha="center", va="center",
                    color="white" if value > (low + 1) / 2 else "black")
    ax.set_title(f"{hotel} — OOF probability correlation")
fig.colorbar(image, ax=axes, shrink=0.85, label="Pearson correlation")
display(fig)
plt.close(fig)

# Section 4 high-error days: the 15 largest RF30 absolute daily errors
# across all H2 Validation fold-days, ranked exactly as in Section 4.4.
h2_models = list(shortlist["H2"])
h2_days = ensemble_daily["H2"].copy()
h2_days["RF30 Absolute Error"] = (
    h2_days["RF30"] - h2_days["Actual Cancellations"]
).abs()
h2_high_error_days = h2_days.nlargest(15, "RF30 Absolute Error").copy()
h2_high_error_days.insert(0, "Fold", h2_high_error_days.fold.map(lambda fold: f"F{fold}"))
print("7.1 — H2 Section 4 high-error days: predicted daily cancellations")
display(h2_high_error_days[
    ["Fold", "StayDate", "N Bookings", "Actual Cancellations", *h2_models]
].round(3))
print("7.1 — H2 Section 4 high-error days: signed error (expected − actual)")
display(h2_high_error_days[["Fold", "StayDate", "Actual Cancellations"]].assign(**{
    f"{label} Error": h2_high_error_days[label] - h2_high_error_days["Actual Cancellations"]
    for label in h2_models
}).round(3))

## 6.2 Simple Averaging Experiments

Test whether equally averaging booking-level probabilities from predefined shortlisted combinations improves Validation Log Loss, daily forecasting accuracy and fold-level stability, without learning any ensemble parameter.

In [ ]:
# Predefined equal-weight combinations only. Booking probabilities are
# averaged first; every ensemble metric is recomputed from those averages.
ensemble_plan = {
    "H1": {
        "H1-E1": ("RF30", "CatBoost"),
        "H1-E2": ("RF30", "XGB30"),
        "H1-E3": ("RW365", "CatBoost"),
        "H1-E4": ("RF30", "RW365"),
    },
    "H2": {
        "H2-E1": ("C2", "TabM-E8"),
        "H2-E2": ("C4", "TabM-E8"),
        "H2-E3": ("C2", "C4"),
        "H2-E4": ("C2", "C4", "TabM-E8"),
    },
}
diagnostic_ensembles = {"H1-E4"}  # similar-model diagnostic comparison

def ensemble_artifact_id(hotel, ensemble_id, members, method_tag=""):
    """Unique _e1 ID naming the ensemble, its method and its components."""
    names = [label.replace("-", "") for label in members]
    return "_".join([hotel, ensemble_id.split("-")[1] + method_tag, *names, "e1"])

ensemble_results = {}
for hotel, plan in ensemble_plan.items():
    for ensemble_id, members in plan.items():
        artifact_id = ensemble_artifact_id(hotel, ensemble_id, members)
        result = run_ensemble(
            experiment_id=artifact_id, hotel=hotel, ensemble_id=ensemble_id,
            method="equal_average",
            components={label: ensemble_components[(hotel, label)] for label in members},
            membership=ensemble_membership, split_config=split_config,
            source_fingerprint=ensemble_source_fingerprint,
            output_root=experiment_root,
        )
        assert result["config"]["evaluation_folds"] == [1, 2, 3, 4]
        assert len(result["predictions"]) == len(ensemble_wide[hotel])
        ensemble_results[(hotel, ensemble_id)] = result
        ensemble_status.append({"Hotel": hotel,
                                "Experiment": f"{ensemble_id} equal average",
                                "Status": result["status"],
                                "Artifact ID": artifact_id})

def ensemble_scores(hotel, candidate, metric):
    """Saved Validation metric by fold for a single model or an ensemble."""
    source = ensemble_results.get((hotel, candidate)) or ensemble_components[(hotel, candidate)]
    frame = source["metrics"]
    selected = frame.loc[frame.split.eq("val")].sort_values("fold")
    return pd.Series(selected[metric].to_numpy(dtype=float),
                     index=selected.fold.astype(int).to_numpy())

def ensemble_mean_sd(scores):
    """Format an equal-fold mean and sample SD."""
    return f"{scores.mean():.4f} ± {scores.std(ddof=1):.4f}"

def best_constituent(hotel, members, folds=(1, 2, 3, 4)):
    """Member with the lowest equal-fold mean Val Log Loss on these folds."""
    return min(members, key=lambda label: ensemble_scores(
        hotel, label, "log_loss"
    )[list(folds)].mean())

performance_rows, loss_rows, nrmse_rows, constituent_rows = [], [], [], []
for hotel, plan in ensemble_plan.items():
    for candidate in [*shortlist[hotel], *plan]:
        members = plan.get(candidate)
        loss = ensemble_scores(hotel, candidate, "log_loss")
        daily = ensemble_scores(hotel, candidate, "daily_nrmse")
        reference = best_constituent(hotel, members) if members else None
        ref_loss = ensemble_scores(hotel, reference, "log_loss") if members else None
        ref_daily = ensemble_scores(hotel, reference, "daily_nrmse") if members else None
        name = {"Hotel": hotel, "Model / Ensemble": candidate}
        performance_rows.append({
            **name, "Members": " + ".join(members) if members else "single model",
            "Val Log Loss": ensemble_mean_sd(loss),
            "Δ vs Best Constituent": loss.mean() - ref_loss.mean() if members else np.nan,
            "Daily nRMSE": ensemble_mean_sd(daily),
            "Daily Bias %": ensemble_mean_sd(ensemble_scores(hotel, candidate, "daily_bias_pct")),
            "ROC-AUC": ensemble_mean_sd(ensemble_scores(hotel, candidate, "roc_auc")),
        })
        for rows, scores, ref_scores in (
            (loss_rows, loss, ref_loss), (nrmse_rows, daily, ref_daily)
        ):
            rows.append({
                **name, **{f"F{fold}": value for fold, value in scores.items()},
                "Mean": scores.mean(), "SD": scores.std(ddof=1),
                "Best Constituent": reference if members else pd.NA,
                "Folds Improved": f"{int((scores < ref_scores).sum())}/4"
                if members else pd.NA,
            })
        if members:
            delta = loss - ref_loss
            constituent_rows.append({
                "Hotel": hotel, "Ensemble": candidate, "Best Constituent": reference,
                "Best Single Log Loss": ref_loss.mean(),
                "Ensemble Log Loss": loss.mean(), "Δ Log Loss": delta.mean(),
                "Paired Δ SD": delta.std(ddof=1),
                "Folds Improved": f"{int((delta < 0).sum())}/4",
                "Best Single Daily nRMSE": ref_daily.mean(),
                "Ensemble Daily nRMSE": daily.mean(),
                "Δ Daily nRMSE": daily.mean() - ref_daily.mean(),
            })

for hotel in ("H1", "H2"):
    select = lambda frame: frame.Hotel.eq(hotel)
    print(f"7.2 — {hotel} equal-weight ensemble performance (four-fold mean ± sample SD)")
    display(pd.DataFrame(performance_rows).loc[select].round(4))
    print(f"7.2 — {hotel} fold-level Validation Log Loss; Folds Improved vs best constituent")
    display(pd.DataFrame(loss_rows).loc[select].round(4))
    print(f"7.2 — {hotel} fold-level Daily nRMSE; Folds Improved vs best-Log-Loss constituent")
    display(pd.DataFrame(nrmse_rows).loc[select].round(4))
    print(f"7.2 — {hotel} ensemble vs best constituent (equal-fold means; paired folds)")
    display(pd.DataFrame(constituent_rows).loc[select].round(4))

## 6.3 Temporal Weighted Averaging

Evaluate whether non-equal two-model weights improve on simple averaging when each target fold's weight is estimated only from earlier-fold OOF predictions whose outcomes were complete before that fold's FitDate.

In [ ]:
# Exploratory: the shortlist uses the four-fold Development CV results of
# 7.2. Fold 1 has no earlier Validation history, so it is never weighted.
weighted_shortlist, shortlist_rows = {}, []
for hotel, plan in ensemble_plan.items():
    eligible = [
        ensemble_id for ensemble_id, members in plan.items()
        if len(members) == 2 and ensemble_id not in diagnostic_ensembles
    ]
    ranked = sorted(eligible, key=lambda ensemble_id: ensemble_scores(
        hotel, ensemble_id, "log_loss"
    ).mean())
    weighted_shortlist[hotel] = ranked[:2]
    for rank, ensemble_id in enumerate(ranked, 1):
        shortlist_rows.append({
            "Hotel": hotel, "Ensemble": ensemble_id,
            "Members": " + ".join(plan[ensemble_id]),
            "Four-fold Mean Val Log Loss": ensemble_scores(
                hotel, ensemble_id, "log_loss"
            ).mean(),
            "Rank": rank,
            "Temporal Weighting": "selected" if rank <= 2 else "not selected",
        })
print("7.3 — exploratory shortlist: two-model, non-diagnostic equal averages ranked by 7.2 mean Val Log Loss")
display(pd.DataFrame(shortlist_rows).round(4))

weight_rows, window_rows = [], []
for hotel, selected in weighted_shortlist.items():
    for ensemble_id in selected:
        members = ensemble_plan[hotel][ensemble_id]
        candidate = f"{ensemble_id} TW"
        artifact_id = ensemble_artifact_id(hotel, ensemble_id, members, "TW")
        result = run_ensemble(
            experiment_id=artifact_id, hotel=hotel, ensemble_id=candidate,
            method="temporal_weighted",
            components={label: ensemble_components[(hotel, label)] for label in members},
            membership=ensemble_membership, split_config=split_config,
            source_fingerprint=ensemble_source_fingerprint,
            output_root=experiment_root,
        )
        ensemble_results[(hotel, candidate)] = result
        ensemble_status.append({"Hotel": hotel,
                                "Experiment": f"{ensemble_id} temporal weighted",
                                "Status": result["status"],
                                "Artifact ID": artifact_id})
        for detail in result["fold_details"]:
            available = not detail["status"].startswith("unavailable")
            if not available:
                ensemble_status.append({
                    "Hotel": hotel,
                    "Experiment": f"{ensemble_id} temporal weighted — F{detail['target_fold']}",
                    "Status": detail["status"], "Artifact ID": artifact_id,
                })
            weight_rows.append({
                "Hotel": hotel, "Ensemble": candidate,
                "Target Fold": f"F{detail['target_fold']}",
                "FitDate": detail["fit_date"],
                "Eligible History": ", ".join(
                    f"F{fold}: {count}"
                    for fold, count in detail.get("history_rows_by_fold", {}).items()
                ) or "none",
                "Historical N": detail["historical_n"],
                "Historical Cancel Rate": detail.get("historical_cancel_rate", np.nan),
                "Model A": detail["model_a"],
                "Model A Weight": detail.get("weight_a", np.nan),
                "Model B": detail["model_b"],
                "Model B Weight": detail.get("weight_b", np.nan),
                "Historical Log Loss": detail.get("historical_log_loss", np.nan),
                "Historical Equal-Weight Log Loss": detail.get(
                    "historical_equal_log_loss", np.nan
                ),
                "Improvement vs Equal": detail.get("improvement_vs_equal", np.nan),
                "Status": detail["status"],
            })

        # Best Single, Equal Average and Temporal Weighted are compared on
        # the identical F2–F4 bookings; four-fold 7.2 means stay separate.
        folds = result["config"]["evaluation_folds"]
        assert folds == [2, 3, 4]
        equal_rows = ensemble_results[(hotel, ensemble_id)]["predictions"]
        equal_rows = equal_rows.loc[equal_rows.fold.isin(folds), keys + ["fold"]]
        pd.testing.assert_frame_equal(
            result["predictions"][keys + ["fold"]].sort_values(keys).reset_index(drop=True),
            equal_rows.sort_values(keys).reset_index(drop=True),
        )
        single = best_constituent(hotel, members, folds)
        single_loss = ensemble_scores(hotel, single, "log_loss")[folds]
        for method, source in (
            (f"Best Single ({single})", single),
            ("Equal Average", ensemble_id),
            ("Temporal Weighted", candidate),
        ):
            loss = ensemble_scores(hotel, source, "log_loss")[folds]
            window_rows.append({
                "Hotel": hotel, "Ensemble": ensemble_id, "Method": method,
                **{f"F{fold} Log Loss": loss[fold] for fold in folds},
                "Mean F2–F4 Log Loss": loss.mean(),
                "Δ vs Best Single": (loss - single_loss).mean()
                if source != single else np.nan,
                "Folds Improved": f"{int((loss < single_loss).sum())}/3"
                if source != single else pd.NA,
                "Mean F2–F4 Daily nRMSE": ensemble_scores(
                    hotel, source, "daily_nrmse"
                )[folds].mean(),
            })

for hotel in ("H1", "H2"):
    select = lambda frame: frame.Hotel.eq(hotel)
    print(f"7.3 — {hotel} forward-only learned weights (exploratory; F1 unavailable)")
    display(pd.DataFrame(weight_rows).loc[select].round(4))
    print(f"7.3 — {hotel} weighted vs simple averaging on identical F2–F4 bookings")
    display(pd.DataFrame(window_rows).loc[select].round(4))

## 7.4 Ensemble Comparison

Consolidate the shortlisted single models, equal-weight ensembles and temporally weighted ensembles by hotel, keeping four-fold and F2–F4 evaluation windows separately labelled.

In [ ]:
# Rows are listed by type, not ranked. Paired differences always use the
# strongest single model on the same evaluation folds.
candidate_types = {
    "H1": {"RF30": "Baseline", "RW365": "RF candidate",
           "CatBoost": "Challenger", "XGB30": "Challenger"},
    "H2": {"RF30": "Baseline", "C2": "RF candidate",
           "C4": "RF candidate", "TabM-E8": "Challenger"},
}
candidates = {
    hotel: [*shortlist[hotel], *ensemble_plan[hotel],
            *(f"{ensemble_id} TW" for ensemble_id in weighted_shortlist[hotel])]
    for hotel in ("H1", "H2")
}

def candidate_type(hotel, candidate):
    """Label single models, equal averages and F2–F4 weighted averages."""
    if candidate in candidate_types[hotel]:
        return candidate_types[hotel][candidate]
    if candidate.endswith(" TW"):
        return "Temporal weighted (exploratory)"
    return "Equal average"

def strongest_single(hotel, folds):
    """Lowest equal-fold mean Val Log Loss among singles on these folds."""
    return best_constituent(hotel, list(shortlist[hotel]), folds)

def window_label(folds):
    return "F1–F4" if list(folds) == [1, 2, 3, 4] else f"F{folds[0]}–F{folds[-1]}"

summary_rows, matrix_rows = [], []
for hotel in ("H1", "H2"):
    for candidate in candidates[hotel]:
        loss = ensemble_scores(hotel, candidate, "log_loss")
        folds = loss.index.tolist()
        reference = strongest_single(hotel, folds)
        ref_loss = ensemble_scores(hotel, reference, "log_loss")[folds]
        is_reference = candidate == reference
        summary_rows.append({
            "Hotel": hotel, "Candidate": candidate,
            "Type": candidate_type(hotel, candidate), "Folds": window_label(folds),
            "Val Log Loss": ensemble_mean_sd(loss),
            "Strongest Single (same folds)": reference,
            "Δ vs Strongest Single": np.nan if is_reference else (loss - ref_loss).mean(),
            "Folds Improved": pd.NA if is_reference
            else f"{int((loss < ref_loss).sum())}/{len(folds)}",
            "Daily nRMSE": ensemble_mean_sd(ensemble_scores(hotel, candidate, "daily_nrmse")),
            "Daily Bias %": ensemble_mean_sd(ensemble_scores(hotel, candidate, "daily_bias_pct")),
            "ROC-AUC": ensemble_mean_sd(ensemble_scores(hotel, candidate, "roc_auc")),
        })
        for metric, metric_name in (
            ("log_loss", "Val Log Loss"), ("daily_nrmse", "Daily nRMSE"),
            ("daily_bias_pct", "Daily Bias %"),
        ):
            scores = ensemble_scores(hotel, candidate, metric)
            matrix_rows.append({
                "Hotel": hotel, "Metric": metric_name, "Candidate": candidate,
                **{f"F{fold}": scores.get(fold, np.nan) for fold in range(1, 5)},
                "Mean": scores.mean(), "SD": scores.std(ddof=1),
                "Mean Folds": window_label(folds),
            })

summary = pd.DataFrame(summary_rows)
matrix = pd.DataFrame(matrix_rows)

def show_matrix(frame):
    """Display N/A where no temporally valid weight exists."""
    shown = frame.drop(columns=["Hotel", "Metric"]).round(4)
    fold_columns = [f"F{fold}" for fold in range(1, 5)]
    shown[fold_columns] = shown[fold_columns].astype(object).where(
        shown[fold_columns].notna(), "N/A"
    )
    display(shown)

for hotel in ("H1", "H2"):
    print(f"7.4 — {hotel} ensemble comparison (mean ± sample SD over the stated folds)")
    display(summary.loc[summary.Hotel.eq(hotel)].round(4))
for metric_name in ("Val Log Loss", "Daily nRMSE", "Daily Bias %"):
    for hotel in ("H1", "H2"):
        print(f"7.4 — {hotel} fold-level {metric_name}")
        show_matrix(matrix.loc[matrix.Hotel.eq(hotel) & matrix.Metric.eq(metric_name)])

# Daily forecasts use the same ArrivalDate-within-ValFold groups as earlier
# sections; ensemble totals are sums of ensemble booking probabilities.
daily_errors = {}
for hotel in ("H1", "H2"):
    daily = ensemble_daily[hotel][["fold", "StayDate", "Actual Cancellations"]].copy()
    for candidate in candidates[hotel]:
        if candidate in shortlist[hotel]:
            expected = ensemble_daily[hotel][candidate]
        else:
            expected = daily[["fold", "StayDate"]].merge(
                ensemble_results[(hotel, candidate)]["predictions"]
                .groupby(["fold", "StayDate"], as_index=False).p_cancel.sum(),
                on=["fold", "StayDate"], how="left", validate="one_to_one",
            ).p_cancel
        daily[candidate] = expected.to_numpy()
    daily_errors[hotel] = daily

h2_columns = list(candidates["H2"])
h2_days = h2_high_error_days[["fold", "StayDate"]].merge(
    daily_errors["H2"], on=["fold", "StayDate"], validate="one_to_one"
)
h2_days.insert(0, "Fold", h2_days.fold.map(lambda fold: f"F{fold}"))
h2_days = h2_days.drop(columns="fold").round(3)
h2_days[h2_columns] = h2_days[h2_columns].astype(object).where(
    h2_days[h2_columns].notna(), "N/A"
)
print("7.4 — H2 Section 4 high-error days: predicted daily cancellations (TW = N/A in F1)")
display(h2_days)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, hotel in zip(axes, ("H1", "H2")):
    four_fold = [c for c in candidates[hotel] if not c.endswith(" TW")]
    daily = daily_errors[hotel]
    errors = [daily[c] - daily["Actual Cancellations"] for c in four_fold]
    ax.boxplot(errors, vert=False, showfliers=True, flierprops={"markersize": 3})
    ax.set_yticks(range(1, len(four_fold) + 1), four_fold)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.invert_yaxis()
    ax.set(xlabel="Daily expected − actual cancellations",
           title=f"{hotel} — daily forecast errors (F1–F4 candidates)")
    ax.grid(axis="x", alpha=0.2)
fig.tight_layout()
display(fig)
plt.close(fig)

# Diagnostic reliability on pooled four-fold OOF rows, Section 5 buckets.
# No calibrator is fitted; this is separate from fold-averaged metrics.
bucket_labels = [f"{10 * i}–{10 * (i + 1)}%" for i in range(10)]
reliability_rows = []
for hotel in ("H1", "H2"):
    chosen = (
        strongest_single(hotel, [1, 2, 3, 4]),
        min(ensemble_plan[hotel], key=lambda c: ensemble_scores(hotel, c, "log_loss").mean()),
    )
    for candidate in chosen:
        if candidate in shortlist[hotel]:
            scored = ensemble_wide[hotel][["y_true", candidate]].rename(
                columns={candidate: "p_cancel"}
            )
        else:
            scored = ensemble_results[(hotel, candidate)]["predictions"][["y_true", "p_cancel"]]
        bucket = np.minimum((scored.p_cancel * 10).astype(int), 9)
        grouped = scored.groupby(bucket).agg(
            N=("y_true", "size"), PredSum=("p_cancel", "sum"),
            ActualSum=("y_true", "sum"),
        ).reindex(range(10), fill_value=0)
        for index, row in grouped.iterrows():
            average = row.PredSum / row.N if row.N else np.nan
            actual = row.ActualSum / row.N if row.N else np.nan
            reliability_rows.append({
                "Hotel": hotel, "Candidate": candidate,
                "Probability Bucket": bucket_labels[index], "N": int(row.N),
                "Avg Pred P": average, "Actual Rate": actual,
                "Gap (pp)": 100 * (average - actual),
            })
reliability = pd.DataFrame(reliability_rows)
for hotel in ("H1", "H2"):
    print(f"7.4 — {hotel} pooled OOF reliability (best single and best equal average by four-fold mean Val Log Loss)")
    display(reliability.loc[reliability.Hotel.eq(hotel)].round(4))
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
for ax, hotel in zip(axes, ("H1", "H2")):
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    for candidate, rows in reliability.loc[
        reliability.Hotel.eq(hotel) & reliability.N.gt(0)
    ].groupby("Candidate", sort=False):
        line, = ax.plot(rows["Avg Pred P"], rows["Actual Rate"], alpha=0.6, label=candidate)
        ax.scatter(rows["Avg Pred P"], rows["Actual Rate"], color=line.get_color(),
                   s=20 + 3 * np.sqrt(rows.N), alpha=0.75)
    ax.set(title=f"{hotel} — pooled OOF reliability (marker size ∝ N)",
           xlim=(0, 1), ylim=(0, 1),
           xlabel="Mean predicted probability")
    ax.legend()
axes[0].set_ylabel("Actual cancellation rate")
fig.tight_layout()
display(fig)
plt.close(fig)

print("7.4 — experiment status and immutable artifact IDs")
display(pd.DataFrame(ensemble_status))

### Ensemble Evaluation — Key Insights & Decisions

**H1 — Resort Hotel**
- **Best Ensemble:** H1-E2 (RF30 + XGB30) achieves the lowest Log Loss (0.2586) and Daily nRMSE (0.9038), outperforming the best single model (RW365).
- **Decision:** Retain H1-E2 as the primary candidate and RW365 as the single-model comparator.

**H2 — City Hotel**
- **Best Ensemble:** H2-E4 (C2 + C4 + TabM) achieves the lowest Log Loss (0.3446), while H2-E1 (C2 + TabM) performs comparably (0.3449) with fewer models.
- **Limitation:** Improvements are modest and inconsistent across folds; probability overprediction remains.
- **Decision:** Retain H2-E4 and H2-E1, with C2 and C4 as single-model comparators.

**Temporal Weighted Averaging**
- Does not outperform equal-weight averaging and shows unstable weights across folds. → **Decision:** Discard weighted averaging.

**Decision: Probability Calibration**
- **H1:** Evaluate H1-E2 and RW365.
- **H2:** Evaluate H2-E4, H2-E1, C2, and C4.
- **Next Step:** Compare Uncalibrated, Sigmoid, and Isotonic using leakage-safe temporal evaluation before Final Model Selection.

# 7. Probability Calibration

## 7.1 Calibration Experiments

Fit fixed Sigmoid and Isotonic mappings from earlier, mature OOF predictions for the six Section 7 candidates. Keep each candidate's saved probabilities as the uncalibrated reference.

In [ ]:
# Load only Gold Development labels and saved Development membership.
# Candidate identity, ensemble components, OOF rows and metrics are checked
# before fitting any mapping. No underlying model or ensemble is refitted.
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from utils.model_calibration import (
    SHORTLIST, METHODS, CLIP_EPS, ISOTONIC_MIN_N,
    ISOTONIC_MIN_PER_CLASS, verify_sources, run_calibration,
)

calibration_membership = ds.dataset(
    split_dir / "membership.parquet", format="parquet"
).to_table(filter=ds.field("Partition") == "Dev").to_pandas()
calibration_labels = pd.concat([
    ds.dataset(
        gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet",
        format="parquet",
    ).to_table(
        columns=keys + ["IsCanceled"],
        filter=ds.field("CutoffDate") < pd.Timestamp(
            split_config["test_start"]
        ).date(),
    ).to_pandas()
    for hotel in ("H1", "H2")
], ignore_index=True)
calibration_source_fingerprint = json.loads(
    (split_dir / "config.json").read_text(encoding="utf-8")
)["source_fingerprint"]
calibration_sources = verify_sources(
    experiment_root=experiment_root,
    membership=calibration_membership,
    labels=calibration_labels,
    split_config=split_config,
    rf_params=model_parameters["RF"],
    source_fingerprint=calibration_source_fingerprint,
)
print("8.1 — Verified shortlist source artifacts")
display(pd.DataFrame([{
    "Hotel": hotel, "Candidate": candidate,
    "Source Artifact": source["artifact_id"],
    "OOF N": len(source["predictions"]),
    "Folds": ", ".join(f"F{fold}" for fold in sorted(
        source["predictions"].fold.unique()
    )),
    "Status": "identity, ensemble mean, cohort and metrics verified",
} for (hotel, candidate), source in calibration_sources.items()]))

calibration_results = {}
calibration_registry_rows = []
calibration_readiness_rows = []
calibration_mapping_rows = []
for (hotel, candidate), source in calibration_sources.items():
    for method in METHODS:
        result = run_calibration(
            hotel=hotel, candidate=candidate, method=method,
            source=source, membership=calibration_membership,
            split_config=split_config, output_root=experiment_root,
        )
        calibration_results[(hotel, candidate, method)] = result
        for detail in result["config"]["fold_details"]:
            fold = int(detail["fold"])
            calibration_registry_rows.append({
                "Hotel": hotel, "Candidate": candidate, "Method": method,
                "Fold": f"F{fold}",
                "Historical N": detail["historical_n"] if fold > 1 else pd.NA,
                "Status": detail["status"],
                "Artifact ID": result["config"]["request"]["experiment_id"],
            })
            if method == "Isotonic" and fold > 1:
                calibration_readiness_rows.append({
                    "Hotel": hotel, "Candidate": candidate,
                    "Target Fold": f"F{fold}",
                    "Historical N": detail["historical_n"],
                    "Cancellations": detail["historical_positive"],
                    "Cancel Rate": detail["historical_rate"],
                    "Historical Folds": len(detail["history_folds"]),
                    "Status": "Sigmoid + Isotonic ready"
                    if detail["isotonic_ready"] else
                    "Isotonic unavailable; see registry",
                })
            if method != "Uncalibrated" and detail["status"] == "evaluated":
                mapping = detail["mapping"]
                calibration_mapping_rows.append({
                    "Hotel": hotel, "Candidate": candidate,
                    "Fold": f"F{fold}", "Method": method,
                    "Historical N": detail["historical_n"],
                    "Mapping Parameters": (
                        f"a={mapping['coefficient']:.4f}, "
                        f"b={mapping['intercept']:.4f}, C=1.0"
                        if method == "Sigmoid" else
                        f"{mapping['n_thresholds']} thresholds; out_of_bounds=clip"
                    ),
                })
calibration_registry = pd.DataFrame(calibration_registry_rows)
calibration_readiness = pd.DataFrame(calibration_readiness_rows)
print("8.1 — Historical OOF readiness (Isotonic: N ≥ 1,000 and ≥ 50 per class)")
display(calibration_readiness.round(4))
print("8.1 — Experiment registry; F1 has no historical calibration fit")
display(calibration_registry[[
    "Hotel", "Candidate", "Method", "Fold", "Historical N", "Status"
]])
print("8.1 — Fitted mapping summary (full thresholds saved in config.json)")
display(pd.DataFrame(calibration_mapping_rows))

## 7.2 Calibration Performance & Reliability

Compare methods on matching F2–F4 validation bookings, recomputing booking and ArrivalDate metrics from saved calibrated probabilities. Inspect reliability, daily fit, and extreme predictions separately for H1 and H2.

In [ ]:
def calibration_folds(hotel, candidate, method):
    """Return actual evaluated folds of one saved calibration run."""
    metrics = calibration_results[(hotel, candidate, method)]["metrics"]
    return sorted(set(metrics.fold.astype(int)) & {2, 3, 4})

def fold_metric(hotel, candidate, method, metric, folds):
    """Read one established metric on explicitly matched Val folds."""
    table = calibration_results[(hotel, candidate, method)]["metrics"]
    values = table.set_index("fold").loc[folds, metric].to_numpy(dtype=float)
    return values

def mean_sample_sd(scores):
    """Format equal-fold mean and sample SD without pooling bookings."""
    return f"{scores.mean():.4f} ± {scores.std(ddof=1):.4f}" if len(scores) > 1 else f"{scores.mean():.4f} ± N/A"

calibration_performance_rows = []
calibration_loss_rows = []
calibration_daily_rows = []
calibration_bias_rows = []
calibration_scored = {}
for hotel, candidates in SHORTLIST.items():
    for candidate in candidates:
        for method in METHODS:
            folds = calibration_folds(hotel, candidate, method)
            if not folds:
                continue
            result = calibration_results[(hotel, candidate, method)]
            scored = result["predictions"].loc[
                lambda frame: frame.fold.isin(folds)
            ].copy()
            calibration_scored[(hotel, candidate, method)] = scored
            base = calibration_results[(hotel, candidate, "Uncalibrated")]
            base_scored = base["predictions"].loc[
                lambda frame: frame.fold.isin(folds)
            ]
            paired = scored[keys + ["fold"]].merge(
                base_scored[keys + ["fold"]], on=keys + ["fold"],
                how="outer", validate="one_to_one", indicator=True,
            )
            assert len(paired) == len(scored) == len(base_scored)
            assert paired._merge.eq("both").all()
            loss = fold_metric(hotel, candidate, method, "log_loss", folds)
            base_loss = fold_metric(
                hotel, candidate, "Uncalibrated", "log_loss", folds
            )
            daily = fold_metric(hotel, candidate, method, "daily_nrmse", folds)
            bias = fold_metric(hotel, candidate, method, "daily_bias_pct", folds)
            auc = fold_metric(hotel, candidate, method, "roc_auc", folds)
            label = ", ".join(f"F{fold}" for fold in folds)
            calibration_performance_rows.append({
                "Hotel": hotel, "Candidate": candidate, "Method": method,
                "Folds": label, "Val N": len(scored),
                "Val Log Loss": mean_sample_sd(loss),
                "Δ vs Uncalibrated": (loss - base_loss).mean()
                if method != "Uncalibrated" else np.nan,
                "Daily nRMSE": mean_sample_sd(daily),
                "Daily Bias %": mean_sample_sd(bias),
                "ROC-AUC": mean_sample_sd(auc),
                "Folds Improved": int((loss < base_loss).sum())
                if method != "Uncalibrated" else pd.NA,
            })
            for rows, values, metric in (
                (calibration_loss_rows, loss, "Log Loss"),
                (calibration_daily_rows, daily, "Daily nRMSE"),
            ):
                rows.append({
                    "Hotel": hotel, "Candidate": candidate,
                    "Method": method, "Metric": metric,
                    **{f"F{fold}": value for fold, value in zip(folds, values)},
                    "Mean": values.mean(),
                    "SD": values.std(ddof=1) if len(values) > 1 else np.nan,
                    "Folds Improved": int((values < (
                        base_loss if metric == "Log Loss" else fold_metric(
                            hotel, candidate, "Uncalibrated", "daily_nrmse", folds
                        )
                    )).sum()) if method != "Uncalibrated" else pd.NA,
                })
            for fold in folds:
                part = scored.loc[scored.fold.eq(fold)]
                calibration_bias_rows.append({
                    "Hotel": hotel, "Candidate": candidate,
                    "Method": method, "Fold": f"F{fold}",
                    "Actual Rate": part.y_true.mean(),
                    "Avg Pred P": part.p_cancel.mean(),
                    "Gap (pp)": 100 * (
                        part.p_cancel.mean() - part.y_true.mean()
                    ),
                })
calibration_performance = pd.DataFrame(calibration_performance_rows)
calibration_fold_loss = pd.DataFrame(calibration_loss_rows)
calibration_fold_daily = pd.DataFrame(calibration_daily_rows)
for hotel in ("H1", "H2"):
    print(f"8.2 — {hotel} performance (equal-fold mean ± sample SD; stated F2–F4 coverage)")
    display(calibration_performance.loc[
        calibration_performance.Hotel.eq(hotel)
    ].round(4))
    print(f"8.2 — {hotel} fold-level Validation Log Loss")
    display(calibration_fold_loss.loc[
        calibration_fold_loss.Hotel.eq(hotel)
    ].drop(columns=["Hotel", "Metric"]).round(4))
    print(f"8.2 — {hotel} fold-level Daily nRMSE")
    display(calibration_fold_daily.loc[
        calibration_fold_daily.Hotel.eq(hotel)
    ].drop(columns=["Hotel", "Metric"]).round(4))

# Existing 10-point buckets; each method creates its own bucket membership.
bucket_labels = [f"{10*i}–{10*(i+1)}%" for i in range(10)]
reliability_rows = []
for (hotel, candidate, method), scored in calibration_scored.items():
    bucket = np.minimum((scored.p_cancel * 10).astype(int), 9)
    grouped = scored.groupby(bucket).agg(
        N=("y_true", "size"), PredSum=("p_cancel", "sum"),
        ActualSum=("y_true", "sum"),
    ).reindex(range(10), fill_value=0)
    for index, row in grouped.iterrows():
        n = int(row.N)
        predicted = row.PredSum / n if n else np.nan
        actual = row.ActualSum / n if n else np.nan
        reliability_rows.append({
            "Hotel": hotel, "Candidate": candidate, "Method": method,
            "Risk Bucket": bucket_labels[index], "Bucket Index": index,
            "N": n, "Avg Pred P": predicted, "Actual Rate": actual,
            "Gap (pp)": 100 * (predicted - actual),
        })
calibration_reliability = pd.DataFrame(reliability_rows)
for hotel in ("H1", "H2"):
    selected = calibration_reliability.loc[
        calibration_reliability.Hotel.eq(hotel)
        & calibration_reliability["Bucket Index"].isin(
            [0, 1, 2, 3] if hotel == "H1" else [2, 3]
        )
    ]
    print(f"8.2 — {hotel} reliability buckets (full 0–100% table expandable below)")
    display(selected.drop(columns="Bucket Index").round(4))
    full = calibration_reliability.loc[
        calibration_reliability.Hotel.eq(hotel)
    ].drop(columns="Bucket Index").round(4)
    display(HTML(
        f"<details><summary>{hotel}: all probability buckets</summary>"
        + full.to_html(index=False) + "</details>"
    ))
    candidates = list(SHORTLIST[hotel])
    rows = int(np.ceil(len(candidates) / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(10, 3.8 * rows),
                             sharex=True, sharey=True)
    panels = np.ravel(axes)
    for ax, candidate in zip(panels, candidates):
        ax.plot([0, 1], [0, 1], "k--", linewidth=1)
        for method in METHODS:
            rows = calibration_reliability.loc[
                calibration_reliability.Hotel.eq(hotel)
                & calibration_reliability.Candidate.eq(candidate)
                & calibration_reliability.Method.eq(method)
                & calibration_reliability.N.gt(0)
            ]
            line, = ax.plot(rows["Avg Pred P"], rows["Actual Rate"],
                            alpha=0.6, label=method)
            ax.scatter(rows["Avg Pred P"], rows["Actual Rate"],
                       color=line.get_color(),
                       s=20 + 3 * np.sqrt(rows.N), alpha=0.75)
        ax.set(title=candidate, xlabel="Mean predicted probability",
               xlim=(0, 1), ylim=(0, 1))
        ax.grid(alpha=0.2)
    panels[0].set_ylabel("Actual cancellation rate")
    panels[-1].legend(loc="lower right", fontsize=8)
    fig.suptitle(f"{hotel} — F2–F4 pooled OOF reliability")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

print("8.2 — Aggregate predicted vs actual cancellation rate by fold")
display(pd.DataFrame(calibration_bias_rows).round(4))
print("8.2 — Daily forecasting metrics by fold")
daily_view = calibration_fold_daily.drop(columns=["Metric", "Mean", "SD", "Folds Improved"])
bias = pd.DataFrame([{
    "Hotel": hotel, "Candidate": candidate, "Method": method,
    **{f"F{fold} Daily Bias %": value for fold, value in zip(
        calibration_folds(hotel, candidate, method),
        fold_metric(hotel, candidate, method, "daily_bias_pct",
                    calibration_folds(hotel, candidate, method)),
    )},
} for (hotel, candidate, method) in calibration_scored])
display(daily_view.merge(bias, on=["Hotel", "Candidate", "Method"],
                         validate="one_to_one").round(4))

# Daily forecasts aggregate booking probabilities by ArrivalDate within
# ValFold. Show the leading ensemble for each hotel on the same fold-days.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (hotel, candidate) in zip(axes, (("H1", "H1-E2"), ("H2", "H2-E4"))):
    for method in METHODS:
        scored = calibration_scored[(hotel, candidate, method)]
        daily = scored.groupby(["fold", "StayDate"]).agg(
            predicted=("p_cancel", "sum"), actual=("y_true", "sum")
        )
        ax.scatter(daily.actual, daily.predicted, s=12, alpha=0.35,
                   label=method)
    limit = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, limit], [0, limit], "k--", linewidth=1)
    ax.set(title=f"{hotel} {candidate}: F2–F4 daily fit",
           xlabel="Actual cancellations", ylabel="Expected cancellations")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
fig.tight_layout()
display(fig)
plt.close(fig)

# Reproduce Section 4's top-15 RF30 daily error days, then retain only
# days in valid calibrated folds for a matched H2-E4 comparison.
from utils.model_ensemble import _check_oof
rf30 = pd.read_parquet(experiment_root / "H2_RF_v2" / "predictions.parquet")
rf30_metrics = pd.read_parquet(experiment_root / "H2_RF_v2" / "metrics.parquet")
_check_oof(rf30, rf30_metrics, "H2", calibration_membership,
           calibration_labels, pd.DataFrame(columns=keys))
rf_days = rf30.groupby(["fold", "StayDate"]).agg(
    actual=("y_true", "sum"), rf_expected=("p_cancel", "sum")
).reset_index()
rf_days["abs_error"] = (rf_days.rf_expected - rf_days.actual).abs()
high_days = rf_days.nlargest(15, "abs_error").loc[
    lambda frame: frame.fold.isin([2, 3, 4]), ["fold", "StayDate", "actual"]
]
for method in METHODS:
    daily = calibration_scored[("H2", "H2-E4", method)].groupby(
        ["fold", "StayDate"], as_index=False
    ).p_cancel.sum().rename(columns={"p_cancel": method})
    high_days = high_days.merge(daily, on=["fold", "StayDate"],
                                how="left", validate="one_to_one")
assert high_days[list(METHODS)].notna().all().all()
for method in METHODS:
    high_days[f"{method} Error"] = high_days[method] - high_days.actual
print("8.2 — H2 previously identified RF30 high-error days in F2–F4")
display(high_days.round(3))

# Section 4.5 thresholds: near-zero <= .01, near-one >= .99.
# Individual negative log probability exposes penalties from wrong extremes.
extreme_rows = []
for (hotel, candidate, method), scored in calibration_scored.items():
    for fold, part in scored.groupby("fold"):
        p = part.p_cancel.to_numpy(float)
        y = part.y_true.to_numpy(int)
        individual_loss = -(y * np.log(p) + (1 - y) * np.log1p(-p))
        detail = calibration_results[(hotel, candidate, method)]["config"][
            "fold_details"
        ][int(fold) - 1]
        extreme_rows.append({
            "Hotel": hotel, "Candidate": candidate, "Method": method,
            "Fold": f"F{fold}", "Near-0": int(np.sum(p <= .01)),
            "Near-1": int(np.sum(p >= .99)),
            "Wrong Extreme": int(np.sum(((p <= .01) & (y == 1))
                                           | ((p >= .99) & (y == 0)))),
            "Raw Exact 0/1": detail["raw_exact_zero"] + detail["raw_exact_one"],
            "Clipped": detail["clip_low_n"] + detail["clip_high_n"],
            "Max Individual Loss": individual_loss.max(),
            "Val Log Loss": individual_loss.mean(),
        })
print("8.2 — Extreme probabilities and numerical clipping (ε = 1e-6)")
display(pd.DataFrame(extreme_rows).round(4))

## 7.3 Calibration Comparison

Consolidate paired F2–F4 effects and keep original four-fold Development CV results in a separate reference group. Document technical readiness without choosing a calibration method or fitting a production mapping.

In [ ]:
master = calibration_performance[[
    "Hotel", "Candidate", "Method", "Folds", "Val Log Loss",
    "Δ vs Uncalibrated", "Daily nRMSE", "Daily Bias %", "ROC-AUC",
    "Folds Improved",
]].rename(columns={"Δ vs Uncalibrated": "Δ Log Loss"})
print("8.3 — Master calibration comparison (paired F2–F4 windows)")
display(master.round(4))

delta_rows, tradeoff_rows = [], []
for hotel, candidates in SHORTLIST.items():
    for candidate in candidates:
        for method in ("Sigmoid", "Isotonic"):
            folds = calibration_folds(hotel, candidate, method)
            if not folds:
                continue
            delta_loss = fold_metric(hotel, candidate, method, "log_loss", folds) - fold_metric(
                hotel, candidate, "Uncalibrated", "log_loss", folds
            )
            delta_daily = fold_metric(hotel, candidate, method, "daily_nrmse", folds) - fold_metric(
                hotel, candidate, "Uncalibrated", "daily_nrmse", folds
            )
            delta_abs_bias = np.abs(fold_metric(
                hotel, candidate, method, "daily_bias_pct", folds
            )) - np.abs(fold_metric(
                hotel, candidate, "Uncalibrated", "daily_bias_pct", folds
            ))
            delta_rows.append({
                "Hotel": hotel, "Candidate": candidate, "Method": method,
                **{f"F{fold} Δ": value for fold, value in zip(folds, delta_loss)},
                "Mean Δ": delta_loss.mean(),
            })
            tradeoff_rows.append({
                "Hotel": hotel, "Candidate": candidate, "Method": method,
                "Δ Log Loss": delta_loss.mean(),
                "Δ Daily nRMSE": delta_daily.mean(),
                "Δ Absolute Daily Bias": delta_abs_bias.mean(),
                "Folds Improved": f"{int((delta_loss < 0).sum())}/{len(folds)}",
            })
print("8.3 — Paired fold Log Loss differences (calibrated minus uncalibrated)")
display(pd.DataFrame(delta_rows).round(4))
for hotel in ("H1", "H2"):
    print(f"8.3 — {hotel} calibration trade-offs on matched folds")
    display(pd.DataFrame(tradeoff_rows).loc[
        lambda frame: frame.Hotel.eq(hotel)
    ].round(4))

# Group 1 uses the original four-fold saved CV metrics. RF30 is an
# additional reference, not one of the six calibration candidates.
original_rows = []
for hotel in ("H1", "H2"):
    reference_id = f"{hotel}_RF_v2"
    reference_predictions = pd.read_parquet(
        experiment_root / reference_id / "predictions.parquet"
    )
    reference_metrics = pd.read_parquet(
        experiment_root / reference_id / "metrics.parquet"
    )
    _check_oof(reference_predictions, reference_metrics, hotel,
               calibration_membership, calibration_labels,
               pd.DataFrame(columns=keys))
    baseline = pd.read_parquet(
        experiment_root / reference_id / "metrics.parquet"
    ).loc[lambda frame: frame.split.eq("val")].sort_values("fold")
    assert baseline.fold.tolist() == [1, 2, 3, 4]
    sources = {"RF30": {"metrics": baseline}, **{
        candidate: calibration_sources[(hotel, candidate)]
        for candidate in SHORTLIST[hotel]
    }}
    for candidate, source in sources.items():
        metrics = source["metrics"]
        validation = metrics.loc[metrics.split.eq("val")].sort_values("fold")
        original_rows.append({
            "Hotel": hotel, "Candidate": candidate, "Folds": "F1–F4",
            "Val Log Loss": mean_sample_sd(validation.log_loss.to_numpy()),
            "Daily nRMSE": mean_sample_sd(validation.daily_nrmse.to_numpy()),
            "Daily Bias %": mean_sample_sd(validation.daily_bias_pct.to_numpy()),
            "ROC-AUC": mean_sample_sd(validation.roc_auc.to_numpy()),
        })
print("8.3 — Group 1: original four-fold Development CV references")
display(pd.DataFrame(original_rows))
print("8.3 — Group 2: temporal calibration evaluation on stated F2–F4 folds")
display(master.round(4))

readiness_rows = []
for (hotel, candidate, method), result in calibration_results.items():
    folds = calibration_folds(hotel, candidate, method)
    readiness_rows.append({
        "Hotel": hotel, "Candidate": candidate,
        "Calibration Method": method,
        "Evaluation Folds": ", ".join(f"F{fold}" for fold in folds),
        "Historical Fit Protocol": "earlier mature OOF by target FitDate"
        if method != "Uncalibrated" else "identity; no fit",
        "Status": "evaluated; selection pending" if folds else "unavailable",
        "Final Mapping": (
            "requires chronological OOS scores from chosen refit; "
            "no Test/OOT labels; compatible with chosen model/ensemble"
        ),
    })
print("8.3 — Technical readiness; fold-specific mappings are not production calibrators")
display(pd.DataFrame(readiness_rows))

# 8. Final Model Selection

## 8.1 Final Candidate Comparison

Compare the original uncalibrated four-fold Development CV results for the shortlisted single models and ensembles, with RF30 as the reference. All values come from verified saved artifacts and use equal-fold means and sample standard deviations.

In [ ]:
# Read Development membership/labels and immutable candidate artifacts.
# This review does not fit a model, create predictions or access Test/OOT.
import json
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from utils.model_calibration import SHORTLIST, verify_sources
from utils.model_ensemble import verify_component

review_membership = ds.dataset(
    split_dir / "membership.parquet", format="parquet"
).to_table(filter=ds.field("Partition") == "Dev").to_pandas()
review_labels = pd.concat([
    ds.dataset(
        gold_root / "model_dataset" / f"gold_model_dataset_{hotel}.parquet",
        format="parquet",
    ).to_table(
        columns=keys + ["IsCanceled"],
        filter=ds.field("CutoffDate") < pd.Timestamp(
            split_config["test_start"]
        ).date(),
    ).to_pandas()
    for hotel in ("H1", "H2")
], ignore_index=True)
review_fingerprint = json.loads(
    (split_dir / "config.json").read_text(encoding="utf-8")
)["source_fingerprint"]
feature_contract = json.loads(
    (gold_root / "feature_contract.json").read_text(encoding="utf-8")
)
binary = development_config["binary_predictors"]
selected_features = {
    "numeric": [name for name in feature_contract["numeric_predictors"]
                if name not in binary],
    "binary": binary,
    "categorical": feature_contract["categorical_predictors"],
}
assert len(sum(selected_features.values(), [])) == 30
review_sources = verify_sources(
    experiment_root=experiment_root, membership=review_membership,
    labels=review_labels, split_config=split_config,
    rf_params=model_parameters["RF"], source_fingerprint=review_fingerprint,
)
rf_estimator = RandomForestClassifier(**model_parameters["RF"])
rf_identity = {
    "model_name": "RF",
    "experiment_version": development_config["experiment_version"],
    "estimator_params": {
        name: "NaN" if isinstance(value, float) and np.isnan(value) else value
        for name, value in rf_estimator.get_params().items()
    },
}
for hotel in ("H1", "H2"):
    review_sources[(hotel, "RF30")] = verify_component(
        experiment_root=experiment_root, artifact_id=f"{hotel}_RF_v2",
        hotel=hotel, identity=rf_identity,
        membership=review_membership, labels=review_labels,
        test_keys=pd.DataFrame(columns=keys),
        selected_features=selected_features, split_config=split_config,
        source_fingerprint=review_fingerprint,
    )
review_order = {
    "H1": ["RF30", "RW365", "H1-E2"],
    "H2": ["RF30", "C2", "C4", "H2-E1", "H2-E4"],
}
assert set(review_sources) == {
    (hotel, candidate) for hotel, candidates in review_order.items()
    for candidate in candidates
}

def review_scores(hotel, candidate, metric, folds=(1, 2, 3, 4)):
    """Read exact saved ValFold metric values in chronological order."""
    frame = review_sources[(hotel, candidate)]["metrics"]
    selected = frame.loc[
        frame.split.eq("val") & frame.fold.isin(folds)
    ].sort_values("fold")
    assert selected.fold.tolist() == list(folds)
    return selected[metric].to_numpy(dtype=float)

def review_mean_sd(values):
    """Format an equal-fold mean and sample SD; no pooled OOF metric."""
    return f"{np.mean(values):.4f} ± {np.std(values, ddof=1):.4f}"

review_numeric = []
review_display = []
for hotel, candidates in review_order.items():
    for candidate in candidates:
        kind = "Baseline" if candidate == "RF30" else (
            "Ensemble" if SHORTLIST[hotel][candidate]["kind"] == "ensemble"
            else "Single"
        )
        scores = {
            name: review_scores(hotel, candidate, metric)
            for name, metric in (
                ("Val Log Loss", "log_loss"),
                ("Daily nRMSE", "daily_nrmse"),
                ("Daily Bias %", "daily_bias_pct"),
                ("ROC-AUC", "roc_auc"),
            )
        }
        review_numeric.append({
            "Hotel": hotel, "Candidate": candidate, "Type": kind,
            **{name: float(values.mean()) for name, values in scores.items()},
            "Log Loss SD": float(scores["Val Log Loss"].std(ddof=1)),
        })
        review_display.append({
            "Hotel": hotel, "Candidate": candidate, "Type": kind,
            **{name: review_mean_sd(values) for name, values in scores.items()},
        })
review_numeric = pd.DataFrame(review_numeric)
for hotel in ("H1", "H2"):
    print(f"9.1 — {hotel} original F1–F4 Development CV (mean ± sample SD)")
    display(pd.DataFrame(review_display).loc[
        lambda frame: frame.Hotel.eq(hotel)
    ].drop(columns="Hotel"))

## 8.2 Single vs Ensemble Trade-off

Measure each ensemble's paired four-fold change relative to the strongest shortlisted single model by Log Loss. Describe the number and identity of component models alongside the daily forecasting trade-off.

In [ ]:
single_references = {}
ensemble_comparisons = {
    "H1": ["H1-E2"], "H2": ["H2-E1", "H2-E4"],
}
best_ensembles = {
    hotel: min(ensembles, key=lambda candidate: review_scores(
        hotel, candidate, "log_loss"
    ).mean())
    for hotel, ensembles in ensemble_comparisons.items()
}
assert best_ensembles == {"H1": "H1-E2", "H2": "H2-E4"}
for hotel, candidates in review_order.items():
    singles = [candidate for candidate in candidates
               if candidate != "RF30"
               and SHORTLIST[hotel][candidate]["kind"] == "single"]
    single_references[hotel] = min(
        singles, key=lambda candidate: review_scores(
            hotel, candidate, "log_loss"
        ).mean()
    )
assert single_references == {"H1": "RW365", "H2": "C4"}

tradeoff_rows, paired_loss_rows, paired_daily_rows = [], [], []
for hotel, ensembles in ensemble_comparisons.items():
    single = single_references[hotel]
    base_loss = review_scores(hotel, single, "log_loss")
    base_daily = review_scores(hotel, single, "daily_nrmse")
    for ensemble in ensembles:
        # Exact outer one-to-one join prevents comparison of different
        # validation bookings or fold/label/date identities.
        left = review_sources[(hotel, single)]["predictions"]
        right = review_sources[(hotel, ensemble)]["predictions"]
        paired = left[keys + [
            "fold", "y_true", "CutoffDate", "StayDate"
        ]].merge(
            right[keys + ["fold", "y_true", "CutoffDate", "StayDate"]],
            on=keys, how="outer", validate="one_to_one",
            suffixes=("_single", "_ensemble"), indicator=True,
        )
        assert len(paired) == len(left) == len(right)
        assert paired._merge.eq("both").all()
        for field in ("fold", "y_true", "CutoffDate", "StayDate"):
            assert paired[f"{field}_single"].eq(
                paired[f"{field}_ensemble"]
            ).all()
        loss = review_scores(hotel, ensemble, "log_loss")
        daily = review_scores(hotel, ensemble, "daily_nrmse")
        delta_loss = loss - base_loss
        delta_daily = daily - base_daily
        tradeoff_rows.append({
            "Hotel": hotel, "Single Reference": single,
            "Ensemble": ensemble,
            "Lowest Ensemble Log Loss": ensemble == best_ensembles[hotel],
            "Δ Log Loss": delta_loss.mean(),
            "Log Loss Improvement %": -100 * delta_loss.mean() / base_loss.mean(),
            "Δ Daily nRMSE": delta_daily.mean(),
            "Daily nRMSE Improvement %": -100 * delta_daily.mean() / base_daily.mean(),
        })
        paired_loss_rows.append({
            "Hotel": hotel, "Ensemble": ensemble,
            **{f"F{fold} Δ Log Loss": value for fold, value in
               enumerate(delta_loss, 1)},
            "Mean Δ": delta_loss.mean(),
            "Folds Improved": f"{int((delta_loss < 0).sum())}/4",
        })
        paired_daily_rows.append({
            "Hotel": hotel, "Ensemble": ensemble,
            **{f"F{fold} Δ Daily nRMSE": value for fold, value in
               enumerate(delta_daily, 1)},
            "Mean Δ": delta_daily.mean(),
            "Folds Improved": f"{int((delta_daily < 0).sum())}/4",
        })
print("9.2 — Ensemble changes vs best shortlisted single (exact four-fold metrics)")
display(pd.DataFrame(tradeoff_rows).round(4))

complexity_rows = []
for hotel, candidates in review_order.items():
    for candidate in candidates:
        if candidate == "RF30":
            continue
        source = review_sources[(hotel, candidate)]
        if SHORTLIST[hotel][candidate]["kind"] == "ensemble":
            members = [part["label"] for part in
                       source["config"]["request"]["components"]]
        else:
            members = [f"{candidate} (RF)"]
        complexity_rows.append({
            "Hotel": hotel, "Candidate": candidate,
            "Type": SHORTLIST[hotel][candidate]["kind"].title(),
            "Number of Component Models": len(members),
            "Components": " + ".join(members),
            "Calibration": "None",
        })
print("9.2 — Descriptive model complexity from saved configurations")
display(pd.DataFrame(complexity_rows))
print("9.2 — Paired fold Validation Log Loss difference vs single")
display(pd.DataFrame(paired_loss_rows).round(4))
print("9.2 — Paired fold Daily nRMSE difference vs single")
display(pd.DataFrame(paired_daily_rows).round(4))

## 8.3 Temporal Stability & Selection Summary

Show four-fold stability and a separately labelled F2–F4 sensitivity view from the same original uncalibrated predictions. Code-generated metric leaders are descriptive; the final model decision remains open.

In [ ]:
stability_rows, sensitivity_rows, evidence_rows = [], [], []
for hotel, candidates in review_order.items():
    for candidate in candidates:
        loss = review_scores(hotel, candidate, "log_loss")
        daily = review_scores(hotel, candidate, "daily_nrmse")
        stability_rows.append({
            "Hotel": hotel, "Candidate": candidate,
            **{f"F{fold} Log Loss": value for fold, value in
               enumerate(loss, 1)},
            "Mean": loss.mean(), "SD": loss.std(ddof=1),
        })
        sensitivity_rows.append({
            "Hotel": hotel, "Candidate": candidate,
            "Log Loss F1–F4": loss.mean(),
            "Log Loss F2–F4": loss[1:].mean(),
            "Daily nRMSE F1–F4": daily.mean(),
            "Daily nRMSE F2–F4": daily[1:].mean(),
        })
        if candidate != "RF30":
            source = review_sources[(hotel, candidate)]
            members = (
                [part["label"] for part in
                 source["config"]["request"]["components"]]
                if SHORTLIST[hotel][candidate]["kind"] == "ensemble"
                else [f"{candidate} (RF)"]
            )
            evidence_rows.append({
                "Hotel": hotel, "Candidate": candidate,
                "Mean Val Log Loss": loss.mean(),
                "Daily nRMSE": daily.mean(),
                "Log Loss SD": loss.std(ddof=1),
                "F2–F4 Log Loss": loss[1:].mean(),
                "Components": " + ".join(members),
            })
stability = pd.DataFrame(stability_rows)
sensitivity = pd.DataFrame(sensitivity_rows)
evidence = pd.DataFrame(evidence_rows)
for hotel in ("H1", "H2"):
    print(f"9.3 — {hotel} original four-fold Validation Log Loss stability")
    display(stability.loc[stability.Hotel.eq(hotel)].drop(columns="Hotel").round(4))
    fig, ax = plt.subplots(figsize=(7, 4))
    for candidate in review_order[hotel]:
        scores = review_scores(hotel, candidate, "log_loss")
        ax.plot(range(1, 5), scores, marker="o", label=candidate,
                linestyle="--" if candidate == "RF30" else "-")
    ax.set(xlabel="Validation fold", ylabel="Validation Log Loss",
           title=f"{hotel} — Original F1–F4 Development CV")
    ax.set_xticks(range(1, 5), [f"F{fold}" for fold in range(1, 5)])
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
    fig.tight_layout()
    display(fig)
    plt.close(fig)

for hotel in ("H1", "H2"):
    print(f"9.3 — {hotel} recent-fold sensitivity (same original OOF predictions)")
    display(sensitivity.loc[sensitivity.Hotel.eq(hotel)].drop(
        columns="Hotel"
    ).round(4))
    print(f"9.3 — {hotel} consolidated shortlisted-candidate evidence")
    display(evidence.loc[evidence.Hotel.eq(hotel)].drop(
        columns="Hotel"
    ).round(4))
    selected = evidence.loc[evidence.Hotel.eq(hotel)]
    leader_rows = []
    for metric, name in (
        ("Mean Val Log Loss", "Lowest F1–F4 Log Loss"),
        ("Daily nRMSE", "Lowest F1–F4 Daily nRMSE"),
        ("Log Loss SD", "Lowest fold-to-fold Log Loss SD"),
        ("F2–F4 Log Loss", "Lowest F2–F4 Log Loss"),
    ):
        best = selected[metric].min()
        leader_rows.append({
            "Metric Leader Only": name,
            "Candidate": ", ".join(selected.loc[
                np.isclose(selected[metric], best, rtol=0, atol=1e-12),
                "Candidate",
            ]),
            "Value": best,
        })
    print(f"9.3 — {hotel} descriptive metric leaders (no selection decision)")
    display(pd.DataFrame(leader_rows).round(4))

# Read Section 8's immutable F2–F4 calibration artifacts. This only
# verifies saved identities/metrics and does not fit any mapping.
calibration_reference_rows = []
for hotel, candidates in SHORTLIST.items():
    for candidate in candidates:
        means = {}
        for method in ("Uncalibrated", "Sigmoid", "Isotonic"):
            run_id = f"{hotel}_{candidate.removeprefix(hotel + '-')}_{method}_p1"
            folder = experiment_root / run_id
            assert {path.name for path in folder.iterdir()} == {
                "config.json", "metrics.parquet", "predictions.parquet"
            }
            config = json.loads((folder / "config.json").read_text(
                encoding="utf-8"
            ))
            request = config["request"]
            assert request["experiment_id"] == run_id
            assert request["candidate"] == candidate
            assert request["method"] == method
            assert request["split_id"] == "split_v2"
            assert request["source_artifact_id"] == review_sources[
                (hotel, candidate)
            ]["artifact_id"]
            assert request["source_fingerprint"] == review_sources[
                (hotel, candidate)
            ]["fingerprint"]
            metrics = pd.read_parquet(folder / "metrics.parquet")
            predictions = pd.read_parquet(folder / "predictions.parquet")
            evaluated = metrics.loc[metrics.fold.isin([2, 3, 4])].sort_values("fold")
            assert evaluated.fold.tolist() == [2, 3, 4]
            scored_window = predictions.loc[predictions.fold.isin([2, 3, 4])]
            original_window = review_sources[(hotel, candidate)][
                "predictions"
            ].loc[lambda frame: frame.fold.isin([2, 3, 4])]
            aligned = scored_window[keys + ["fold", "y_true", "p_original"]].merge(
                original_window[keys + ["fold", "y_true", "p_cancel"]],
                on=keys, how="outer", validate="one_to_one",
                suffixes=("", "_source"), indicator=True,
            )
            assert len(aligned) == len(scored_window) == len(original_window)
            assert aligned._merge.eq("both").all()
            assert aligned.fold.eq(aligned.fold_source).all()
            assert aligned.y_true.eq(aligned.y_true_source).all()
            assert np.array_equal(aligned.p_original, aligned.p_cancel)
            for fold in (2, 3, 4):
                scored = predictions.loc[predictions.fold.eq(fold)]
                row = evaluated.loc[evaluated.fold.eq(fold)].iloc[0]
                assert np.isclose(row.log_loss, log_loss(
                    scored.y_true, scored.p_cancel, labels=[0, 1]
                ))
            means[method] = evaluated.log_loss.mean()
        calibration_reference_rows.append({
            "Hotel": hotel, "Candidate": candidate,
            "Uncalibrated F2–F4": means["Uncalibrated"],
            "Sigmoid Δ Log Loss": means["Sigmoid"] - means["Uncalibrated"],
            "Isotonic Δ Log Loss": means["Isotonic"] - means["Uncalibrated"],
            "Status": "no mean Log Loss improvement" if (
                means["Sigmoid"] >= means["Uncalibrated"]
                and means["Isotonic"] >= means["Uncalibrated"]
            ) else "review mixed results",
        })
print("9.3 — Section 8 calibration reference (F2–F4 only; read-only)")
display(pd.DataFrame(calibration_reference_rows).round(4))

### Final Model Selection — Key Insights

- **H1 — Resort Hotel:** H1-E2 (**RF30 + XGB30, equal-weight ensemble**) outperforms the best single model, RW365 (**RF30 with 365-day recency weighting**), in both Log Loss (0.2586 vs 0.2612) and Daily nRMSE (0.9038 vs 0.9534), improving Log Loss across 3/4 folds and reducing Daily nRMSE by approximately 5.2%.

- **H2 — City Hotel:** H2-E4 (**C2 + C4 + TabM-E8, equal-weight ensemble**) achieves the lowest Log Loss (0.3446), but only marginally outperforms H2-E1 (**C2 + TabM-E8, equal-weight ensemble**; 0.3449). H2-E1 offers better Daily nRMSE (0.9244 vs 0.9316) with fewer component models (2 vs 3). Meanwhile, C2 (**RF30 with 180-day recency weighting + FE4 historical segment cancellation rate**) demonstrates greater fold-level stability and achieves the lowest Log Loss on F2–F4. Here, C4 is a regularized RF with 90-day recency weighting, and TabM-E8 is TabM trained for 8 epochs.